# Systemic Discovery of Phage-Encoded Inhibitors in Oral Pathogens
---

Executive Summary: 

To develop a systematic computation pipeline for identifying inhibitors of bacteria that causes dental diseases. The system should utilize parallel computing with cuda/cudf for highly efficient data search.

Workflow: 

0) collect phage library

1) generates + curates a candidate library processed with sequence clustering and flitering with cuDF
2) uses AlphaFold3 complex prediction for high-confidence shortlist candidates (expensive, accurate)
3) uses fast docking or a learned scoring function for broad screening (cheap, approximate)
4) feeds docking hits back into AlphaFold3 for refined complex prediction


More Detailed Workflow:

Stage 1 — Generate novel candidates (variant generation). You take the propeptide template(s) you just pulled and computationally generate large numbers of variants from it — point mutations, saturation mutagenesis at key contact residues, or fragment recombination. You don't know yet which of these variants would actually inhibit gingipains; you're just generating a large candidate pool of plausible sequences, most of which will be junk. This is the step you flagged earlier as "Tier 2" expansion, and it's the next thing to build, not something you have running yet. Alongside of this also collect existing sequences. 

Stage 2 — Cheap filtering before expensive prediction. Before folding thousands of candidates, you cluster them (MMseqs2-GPU) to collapse near-duplicates and use cuDF to filter out obviously bad candidates (wrong length, degenerate sequence, poor cluster confidence). This doesn't tell you which ones inhibit gingipains — it just trims the pool down to something AlphaFold3 can afford to process.

Stage 3 — You run AlphaFold3 in complex-prediction mode, feeding it both the gingipain catalytic domain (your target structure) and each surviving candidate variant together, and it predicts whether/how they'd physically bind as a complex. AlphaFold3 outputs confidence metrics for that predicted interaction — ipTM and pTM specifically measure how confident the model is that the two molecules form a real, stable interface at that geometry. 

You rank all your candidates by these scores. The candidates that score highest — strong predicted binding to the gingipain active site, ideally better than what the native propeptide itself would score — are your discovered inhibitor candidates. That ranked list is the actual output of the pipeline.

# Stage 1. Target Justification

**Goal: Establish which defense systems in which oral pathogens are worth targeting**

This workflow is suitable targeting pathogens that have defense systems in relation to the generation of proteins (e.g. protease enzymes) that can be inhibited with recombinant forms of peptides binding to their active site. 

A similar analogy is of a key and lock situation - the workflow aims to find copies of keys that can fit into the lock, disallowing the originally intended key from binding to the lock and initiating further action.


**Example Target: P. gingivalis**

<figure style="text-align: center;">
  <img src=images/gingipains.png width="30%" alt="Centered and small image">
  <figcaption>Interaction of gingipain catalytic domains with their propeptides. [<a href="https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0065447">6</a>] </figcaption>
</figure>


One of its key defense system is **Enzymatic "Shields" (Gingipains)**: It produces powerful cysteine protease enzymes called gingipains (RgpA, RgpB, and Kgp). These act as a defense against the human immune system by degrading complement factors (C3, C4, C5) and antibodies (IgG), effectively disarming the host's ability to kill the bacteria. [[1](https://pmc.ncbi.nlm.nih.gov/articles/PMC3894813/), [2](https://pmc.ncbi.nlm.nih.gov/articles/PMC3463344/), [3](https://pmc.ncbi.nlm.nih.gov/articles/PMC12385285/)]

Inhibition: The bacterium's own enzymes are synthesized with an attached "propeptide" region that keeps them inactive until they are secreted. Recombinant forms of these propeptides have been shown to bind to and inhibit the active gingipains. [[4](https://pmc.ncbi.nlm.nih.gov/articles/PMC3677877/), [5](https://pmc.ncbi.nlm.nih.gov/articles/PMC10403534/)]



# Stage 2. Building Phage Protein Library

**Goal: A filtered, clustered database of candidate inhibitor sequences**

**Environment Setup**

Recommend: create a new environment with a Python version that is RAPIDS compatiable, e.g. 3.11 - 3.14

In [ ]:
# For mac0S/Linux
# To create a new virtual environment named myvenv with Python version 3.11.15
!python3.11.15 -m venv myvenv
# activate venv
!source /path/to/venv

# check Nvidia cuda version (for Nvidia GPU)
!nvcc --version
# version is what proceeds after Build cuda_1x.x

Install Docker and NVIDIA support for Docker using [these instructions](https://github.com/google-deepmind/alphafold3/blob/main/docs/installation.md#installing-docker).

Install RAPIDS following your cuda and Python version using this [selector tool](https://docs.rapids.ai/install/#:~:text=Next%20Steps-,Install%20RAPIDS,-Use%20the%20selector).

In [ ]:
# E.g. RAPIDS installation command for cuda 12 and Python 3.11.x

docker run --gpus all --pull always --rm -it \
    --shm-size=1g --ulimit memlock=-1 --ulimit stack=67108864 \
    nvcr.io/nvidia/rapidsai/base:26.06-cuda12-py3.11

## Stage 2a. Phage Database Selection and Download

Example Selection Tierlist:

- **Tier 1 (validate the approach, ~50–200 sequences):** known/close-homolog propeptides and any published gingipain inhibitor peptides like MEROPS. Fold these first — if your docking/ranking pipeline correctly recovers the known actives near the top, you've validated the method before scaling up.

- **Tier 2 (broaden, ~1,000–10,000):** UniProt search for cysteine protease propeptide domains genome-wide (not just MEROPS-curated), or peptide fragments generated by computational mutagenesis/scanning of the Tier 1 hits.
- **Tier 3 (large-scale, only if Tier 1–2 justify it):** ZINC or ChEMBL small-molecule sets if you want to move beyond peptide inhibitors into drug-like chemical space — this is where you'd actually need the storage and GPU scale, since these libraries run into the millions.


### Dataset 1: [MEROPS](https://www.ebi.ac.uk/merops/download_list.shtml)

Datasets to Download:

- Family Protein Sequences (#6, [C25.lib](https://ftp.ebi.ac.uk/pub/databases/merops/current_release/seqlib/c25.lib)): Go to the seqlib directory listing and pull the file for family C25 (clan CD, the gingipain/caspase-like cysteine protease family). That gives you a FASTA of peptidase and inhibitor units specific to that family.

- Peptidase and Inhibitor Accession Numbers (#2, [dnld_list.txt](https://ftp.ebi.ac.uk/pub/databases/merops/current_release/dnld_list.txt)): useful if you want to cross-reference accessions with organism/family identifiers before fetching sequences elsewhere (e.g. to pull full UniProt entries with domain annotations rather than just the trimmed peptidase unit).

- pepunit.lib (#3) only if the C25 family file isn't granular enough and you need to grep it yourself — it's non-redundant peptidase/inhibitor units only (propeptide + catalytic domain region, not full-length), which is actually a good thing for you since the propeptide is exactly the inhibitory element you care about.

What the code is doing: 

It reads through every entry in the C25.lib file (the MEROPS-curated cysteine protease family that gingipains belong to), and for each entry it pulls apart the header text to figure out three things — which protein it is, what organism it's from, and whether the included sequence fragment represents the enzyme's catalytic region ("peptidase unit") or a separate inhibitory region ("propeptide unit"/"inhibitor unit"). It sorts these into separate files, and additionally tags anything from P. gingivalis specifically into its own file. When the propeptide-unit count comes back too low (a set number, e.g. 5), it falls back to fetching full protein records directly from UniProt and cutting out the propeptide region using UniProt's own domain annotations.

In [ ]:
#!/usr/bin/env python3
"""
Parse MEROPS C25.lib + dnld_list.txt to build:
  1. catalytic_domains.fasta   - peptidase units (docking targets / negative controls)
  2. propeptide_candidates.fasta - propeptide units (Tier 1 inhibitor seeds)
  3. pgingivalis_only.fasta    - either pool restricted to P. gingivalis (taxid 5911)

If the propeptide pool comes back too small, optionally fetches full UniProt
entries for known gingipain accessions and excises the propeptide region
using UniProt's own Propeptide feature annotation (FT lines).

Usage:
    python parse_merops_c25.py C25.lib dnld_list.txt [--uniprot-fallback] [--min-propeptides N]
    
"""

import re
import sys
import time
import argparse
from pathlib import Path

try:
    import requests
except ImportError:
    requests = None

UNIPROT_REST = "https://rest.uniprot.org/uniprotkb"

# Known gingipain UniProt accessions (RgpA, RgpB, Kgp - P. gingivalis W83/ATCC 33277).
# Extend this list if you identify more strain-specific entries from your MEROPS pull.
GINGIPAIN_UNIPROT_ACCESSIONS = [
    "P28784",  # RgpA, CPGR_PORGI / similar - VERIFY against your organism card before use
    "P95493",  # RgpB - VERIFY
    "Q51817",  # Kgp - VERIFY
]

PGINGIVALIS_TAXID = "5911"

HEADER_RE = re.compile(   
    r"^>(?P<acc>MER\d+)\s*-\s*(?P<desc>.*?)\s*\[(?P<family>C25\.\w+)\]"
    r"#C25#\{(?P<unit_type>peptidase unit|propeptide unit|inhibitor unit)\s*:\s*"
    r"(?P<range>\d+-\d+(?:\s*,\s*\d+-\d+)*)\}~source\s+(?P<source>\S+)~"
)

ORG_RE = re.compile(r"\((?P<organism>[^()]+)\)\s*$")


def parse_fasta(path):
    """Yield (header, sequence) tuples from a FASTA file."""
    header, seq_lines = None, []
    with open(path) as fh:
        for line in fh:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if header is not None:
                    yield header, "".join(seq_lines)
                header = line
                seq_lines = []
            else:
                seq_lines.append(line)
    if header is not None:
        yield header, "".join(seq_lines)


def load_taxid_map(dnld_list_path):
    """Map accession (no db prefix) -> taxid from dnld_list.txt."""
    acc_to_taxid = {}
    with open(dnld_list_path) as fh:
        for line in fh:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) < 3:
                continue
            db_acc, family, taxid = parts[0], parts[1], parts[2]
            acc = db_acc.split(":", 1)[-1]  # strip "Trembl:" / "Sprot:" / "MEROPS:" prefix
            acc_to_taxid[acc] = taxid
    return acc_to_taxid


def fetch_uniprot_entry(accession):
    """Fetch a UniProt entry as flat text (includes FT Propeptide lines + sequence)."""
    if requests is None:
        raise RuntimeError("requests library not installed; pip install requests")
    url = f"{UNIPROT_REST}/{accession}.txt"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    return resp.text


def parse_uniprot_flatfile(text):
    """
    Extract (start, end) ranges for PROPEP features and the full sequence
    from a UniProt flat-file (.txt) record.
    Returns (sequence, [(start, end), ...]).
    """
    propep_ranges = []
    seq_lines = []
    in_seq = False

    for line in text.splitlines():
        if line.startswith("FT   PROPEP"):
            # e.g. "FT   PROPEP          20..200"
            m = re.search(r"(\d+)\.\.(\d+)", line)
            if m:
                propep_ranges.append((int(m.group(1)), int(m.group(2))))
        elif line.startswith("SQ   SEQUENCE"):
            in_seq = True
            continue
        elif line.startswith("//"):
            in_seq = False
        elif in_seq:
            seq_lines.append(line.replace(" ", "").strip())

    sequence = "".join(seq_lines)
    return sequence, propep_ranges


def uniprot_fallback(accessions, out_dir, delay=1.0):
    """
    For each accession, fetch the full entry, excise propeptide region(s)
    using UniProt's own FT PROPEP annotation, write to fallback FASTA.
    """
    out_path = out_dir / "uniprot_fallback_propeptides.fasta"
    written = 0
    with open(out_path, "w") as out_fh:
        for acc in accessions:
            try:
                text = fetch_uniprot_entry(acc)
            except Exception as e:
                print(f"  [skip] {acc}: fetch failed ({e})")
                continue

            sequence, propep_ranges = parse_uniprot_flatfile(text)
            if not sequence or not propep_ranges:
                print(f"  [skip] {acc}: no sequence or no PROPEP annotation found")
                continue

            for i, (start, end) in enumerate(propep_ranges, 1):
                # UniProt coordinates are 1-based inclusive
                frag = sequence[start - 1:end]
                if not frag:
                    continue
                out_fh.write(f">{acc}|UniProt_fallback|propeptide_unit_{i}|{start}-{end}\n{frag}\n")
                written += 1

            time.sleep(delay)  # be polite to the API

    print(f"UniProt fallback: wrote {written} propeptide fragment(s) to {out_path}")
    return written


def main():
    # ============================================================
    # CONFIG 
    # ============================================================
    LIB_PATH = "../datasets/MEROPS/MEROPS_C25.lib"              # path to your downloaded C25.lib
    DNLD_PATH = "../datasets/MEROPS/MEROPS_dnld_list.txt"       # path to your downloaded dnld_list.txt
    UNIPROT_FALLBACK_FORCE = False    # set True to always run UniProt fallback
    MIN_PROPEPTIDES = 5               # auto-trigger fallback if MEROPS count is below this
    UNIPROT_ACCESSIONS = GINGIPAIN_UNIPROT_ACCESSIONS  # or override: ["P95493", "Q51817", ...]
    # ============================================================

    lib_path, dnld_path = LIB_PATH, DNLD_PATH
    acc_to_taxid = load_taxid_map(dnld_path)

    out_dir = Path("output/merops_parsed")
    out_dir.mkdir(exist_ok=True)

    catalytic_fh = open(out_dir / "catalytic_domains.fasta", "w")
    propeptide_fh = open(out_dir / "propeptide_candidates.fasta", "w")
    pgingivalis_fh = open(out_dir / "pgingivalis_all.fasta", "w")
    unmatched_fh = open(out_dir / "unmatched_headers.txt", "w")

    counts = {"peptidase unit": 0, "propeptide unit": 0, "inhibitor unit": 0,
              "pgingivalis": 0, "unmatched": 0, "total": 0}

    for header, seq in parse_fasta(lib_path):
        counts["total"] += 1
        m = HEADER_RE.match(header)
        if not m:
            unmatched_fh.write(header + "\n")
            counts["unmatched"] += 1
            continue

        unit_type = m.group("unit_type")
        acc = m.group("acc")
        desc = m.group("desc")
        source = m.group("source")
        org_match = ORG_RE.search(desc)
        organism = org_match.group("organism") if org_match else ""

        record = f">{acc}|{m.group('family')}|{unit_type.replace(' ', '_')}|{source}|{organism}\n{seq}\n"

        if unit_type == "peptidase unit":
            catalytic_fh.write(record)
            counts["peptidase unit"] += 1
        elif unit_type in ("propeptide unit", "inhibitor unit"):
            propeptide_fh.write(record)
            counts[unit_type] += 1

        # organism-name match as primary filter (most reliable here since
        # dnld_list.txt keys are UniProt accessions, not MER IDs)
        is_pgingivalis = "gingivalis" in organism.lower()
        # secondary cross-check via taxid map using the source mnemonic, if present
        if source in acc_to_taxid and acc_to_taxid[source] == PGINGIVALIS_TAXID:
            is_pgingivalis = True

        if is_pgingivalis:
            pgingivalis_fh.write(record)
            counts["pgingivalis"] += 1

    for fh in (catalytic_fh, propeptide_fh, pgingivalis_fh, unmatched_fh):
        fh.close()

    print("Parsed C25.lib:")
    print(f"  total entries:        {counts['total']}")
    print(f"  peptidase units:      {counts['peptidase unit']}")
    print(f"  propeptide units:     {counts['propeptide unit']}")
    print(f"  inhibitor units:      {counts['inhibitor unit']}")
    print(f"  P. gingivalis (any):  {counts['pgingivalis']}")
    print(f"  unmatched headers:    {counts['unmatched']}  (see unmatched_headers.txt)")
    print(f"\nOutput written to: {out_dir.resolve()}")

    merops_propeptide_total = counts["propeptide unit"] + counts["inhibitor unit"]
    should_fallback = UNIPROT_FALLBACK_FORCE or merops_propeptide_total < MIN_PROPEPTIDES

    if should_fallback:
        reason = "forced via UNIPROT_FALLBACK_FORCE" if UNIPROT_FALLBACK_FORCE else \
                f"MEROPS propeptide count ({merops_propeptide_total}) below threshold ({MIN_PROPEPTIDES})"
        print(f"\nTriggering UniProt fallback: {reason}")
        print("NOTE: verify GINGIPAIN_UNIPROT_ACCESSIONS at the top of this script against")
        print("      the organism card on MEROPS / UniProt before trusting these results -")
        print("      strain-specific accessions (W83 vs ATCC 33277) may differ.")
        uniprot_fallback(UNIPROT_ACCESSIONS, out_dir)
    else:
        print(f"\nMEROPS propeptide pool ({merops_propeptide_total}) meets threshold "
            f"({MIN_PROPEPTIDES}) - skipping UniProt fallback. "
            f"Set UNIPROT_FALLBACK_FORCE = True to force it anyway.")


if __name__ == "__main__":
    main()
    
# a peptidase unit (the catalytic domain that does the cutting — target structure for docking/complex prediction)
# a propeptide unit (the N-terminal segment that naturally folds back and blocks that same enzyme's own active site before it's activated — this is the inhibitor scaffold).

# target: find something that binds to the peptidase and so blocks it from binding to the propeptide
# scaffold: using variations of the propeptide units or from natural sequence

Parsed C25.lib:
  total entries:        1187
  peptidase units:      1187
  propeptide units:     0
  inhibitor units:      0
  P. gingivalis (any):  30
  unmatched headers:    0  (see unmatched_headers.txt)

Output written to: /home/ubuntu/tiffany/code/output/merops_parsed

Triggering UniProt fallback: MEROPS propeptide count (0) below threshold (5)
NOTE: verify GINGIPAIN_UNIPROT_ACCESSIONS at the top of this script against
      the organism card on MEROPS / UniProt before trusting these results -
      strain-specific accessions (W83 vs ATCC 33277) may differ.
  [skip] Q9R6S1: no sequence or no PROPEP annotation found
UniProt fallback: wrote 2 propeptide fragment(s) to output/merops_parsed/uniprot_fallback_propeptides.fasta


### Extracting Gingipain Propeptide Sequences

In [2]:
#!/usr/bin/env python3
"""
extract_propeptides.py
======================
Extracts propeptide sequences for RgpA, RgpB, and Kgp from locally-downloaded
UniProt FASTA files, using verified PROPEP boundary coordinates.

Why not MEROPS? MEROPS C25 has zero curated propeptide-unit entries — confirmed.
We go directly to the full-length UniProt precursor sequences and slice out the
propeptide region using UniProt-annotated boundaries.

Outputs:
  propeptide_seeds.fasta   — one canonical propeptide per enzyme (PE=1, reviewed)
  propeptide_all.fasta     — propeptides from all strain variants in the input files

Usage (edit CONFIG block below, then just run):
  python extract_propeptides.py
"""

from pathlib import Path

# ============================================================
# CONFIG — edit these paths to point to your downloaded files
# ============================================================
FASTA_FILES = [
    "../datasets/UniProt/uniprotkb_gingipain_RgpA_Porphyromonas_2026_07_02.fasta",
    "../datasets/UniProt/uniprotkb_gingipain_RgpB_Porphyromonas_2026_07_02.fasta",
    "../datasets/UniProt/uniprotkb_gingipain_Kgp_Porphyromonas_g_2026_07_02.fasta",
]
OUT_DIR = Path("output/propeptides")
# ============================================================

# PROPEP boundaries (1-based, inclusive) verified from UniProt PTM/Processing tab.
# Format: accession -> (propep_start, propep_end, enzyme_label)
#
# P28784 (RgpA): CONFIRMED from UniProt page — Propeptide 25-227
# P95493 (RgpB): INFERRED — signal peptide ~24aa, catalytic domain begins at 230
#                (commercial recombinant proteins expressed from aa 230 confirm boundary)
#                → Propeptide 25-229. GO VERIFY at uniprot.org/P95493 → PTM/Processing tab.
# Q51817 (Kgp):  INFERRED — same signal peptide length pattern (~24aa), propeptide ~200aa
#                → Propeptide 25-227. GO VERIFY at uniprot.org/Q51817 → PTM/Processing tab.
#
# To verify: open each UniProt entry, click "PTM/Processing" section,
# look for the "Propeptide" row and read off POSITION(S).
PROPEP_COORDS = {
    "P28784": (25, 227, "RgpA"),   # CONFIRMED
    "P95493": (25, 229, "RgpB"),   # INFERRED — verify before publication
    "Q51817": (25, 228, "Kgp"),    # INFERRED — verify before publication
    # Strain variants — same boundaries assumed (sequences are highly similar)
    "B2RM93": (21, 224, "RgpA_ATCC33277"),  # ATCC 33277 strain, 1aa shorter signal
    "B2RKU0": (25, 229, "RgpB_ATCC33277"),
    "P72194": (25, 228, "Kgp_ATCC33277"),
}

# Only these accessions go into the "seeds" (canonical, PE=1 reviewed) file
CANONICAL_ACCESSIONS = {"P28784", "P95493", "Q51817"}


def parse_fasta(path):
    """Yield (accession, full_header, sequence) from a UniProt FASTA file."""
    header, seq_lines = None, []
    with open(path) as fh:
        for line in fh:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if header is not None:
                    yield _parse_acc(header), header, "".join(seq_lines)
                header = line
                seq_lines = []
            else:
                seq_lines.append(line.strip())
    if header is not None:
        yield _parse_acc(header), header, "".join(seq_lines)


def _parse_acc(header):
    """Extract accession from UniProt FASTA header e.g. >sp|P28784|CPG1_PORGN ..."""
    parts = header.lstrip(">").split("|")
    return parts[1] if len(parts) >= 2 else parts[0].split()[0]


def slice_propeptide(sequence, start, end, acc):
    """Slice propeptide region (1-based inclusive). Validates length sanity."""
    frag = sequence[start - 1:end]
    if len(frag) < 50:
        print(f"  [warn] {acc}: propeptide slice only {len(frag)} aa — check coordinates")
    if len(frag) > 350:
        print(f"  [warn] {acc}: propeptide slice is {len(frag)} aa — unusually long, check coordinates")
    return frag


def main():
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    seen_accessions = set()
    seeds = []    # canonical PE=1 entries only
    all_propep = []  # every strain variant

    for fasta_path in FASTA_FILES:
        path = Path(fasta_path)
        if not path.exists():
            print(f"[skip] file not found: {fasta_path}")
            continue

        print(f"Reading {path.name} ...")
        for acc, header, seq in parse_fasta(path):
            if acc in seen_accessions:
                continue  # deduplicate — files overlap
            seen_accessions.add(acc)

            if acc not in PROPEP_COORDS:
                # Unknown accession — skip silently (strain variants not in our table)
                continue

            start, end, label = PROPEP_COORDS[acc]

            if end > len(seq):
                print(f"  [warn] {acc}: sequence length {len(seq)} shorter than PROPEP end {end} — skipping")
                continue

            propep_seq = slice_propeptide(seq, start, end, acc)
            if not propep_seq:
                continue

            record = f">{acc}|propeptide|{label}|{start}-{end}\n{propep_seq}\n"
            all_propep.append(record)
            if acc in CANONICAL_ACCESSIONS:
                seeds.append(record)
            print(f"  {acc} ({label}): extracted {len(propep_seq)} aa propeptide [{start}-{end}]")

    # Write outputs
    seeds_path = OUT_DIR / "propeptide_seeds.fasta"
    all_path = OUT_DIR / "propeptide_all.fasta"

    with open(seeds_path, "w") as fh:
        fh.writelines(seeds)
    with open(all_path, "w") as fh:
        fh.writelines(all_propep)

    print(f"\nDone.")
    print(f"  Canonical seeds:  {len(seeds)} sequences  →  {seeds_path}")
    print(f"  All variants:     {len(all_propep)} sequences  →  {all_path}")
    print(f"\nNEXT STEPS:")
    print(f"  1. Verify P95493 and Q51817 coordinates at uniprot.org (PTM/Processing tab)")
    print(f"  2. Feed propeptide_seeds.fasta into variant generation (mutagenesis expansion)")
    print(f"  3. Cluster with MMseqs2-GPU, filter with cuDF, rank with AlphaFold3 ipTM/pTM")


if __name__ == "__main__":
    main()

Reading uniprotkb_gingipain_RgpA_Porphyromonas_2026_07_02.fasta ...
  P95493 (RgpB): extracted 205 aa propeptide [25-229]
  P28784 (RgpA): extracted 203 aa propeptide [25-227]
  B2RKU0 (RgpB_ATCC33277): extracted 205 aa propeptide [25-229]
  B2RM93 (RgpA_ATCC33277): extracted 204 aa propeptide [21-224]
Reading uniprotkb_gingipain_RgpB_Porphyromonas_2026_07_02.fasta ...
Reading uniprotkb_gingipain_Kgp_Porphyromonas_g_2026_07_02.fasta ...
  P72194 (Kgp_ATCC33277): extracted 204 aa propeptide [25-228]
  Q51817 (Kgp): extracted 204 aa propeptide [25-228]

Done.
  Canonical seeds:  3 sequences  →  output/propeptides/propeptide_seeds.fasta
  All variants:     6 sequences  →  output/propeptides/propeptide_all.fasta

NEXT STEPS:
  1. Verify P95493 and Q51817 coordinates at uniprot.org (PTM/Processing tab)
  2. Feed propeptide_seeds.fasta into variant generation (mutagenesis expansion)
  3. Cluster with MMseqs2-GPU, filter with cuDF, rank with AlphaFold3 ipTM/pTM


## Stage 2bi. Generating Point-Substitution Variants of Gingipain Propeptides

**What this does**

For each of the 3 seed sequences, it walks every position and substitutes all 19 alternate amino acids, yielding roughly 11,628 variants total. Each variant gets a header that fully encodes its provenance — accession, the mutation in standard notation (e.g. Y25A means tyrosine at position 25 substituted with alanine), the enzyme, and the original coordinate range. It also writes a TSV with one row per variant, which is what the cuDF filtering step will consume directly.

**A few design decisions worth knowing**

The SKIP_POSITIONS dict is currently empty, meaning every position gets mutated. Once you have AlphaFold3 results back, you'll identify residues at the active-site interface that are critical for binding — you can protect those positions by adding them to SKIP_POSITIONS and re-running, which focuses the library on the periphery where substitutions are more tolerated.
The wild-type seeds are included in the output (controlled by INCLUDE_WILDTYPE) so the clustering and filtering steps have a reference point.
Expected runtime — pure Python string operations on ~11k sequences, should finish in a few seconds.
Run it with SEEDS_FASTA pointing to your propeptide_seeds.fasta, then come back and we'll write the cuDF filtering script next.

In [ ]:
#!/usr/bin/env python3
"""
generate_variants.py
====================
Generates single-point substitution variants from propeptide seed sequences.

Each seed produces (L × 19) variants where L = sequence length and 19 = all
standard amino acids minus the wild-type residue at that position.

With 3 seeds of ~204 aa each: ~3 × 204 × 19 ≈ 11,628 variants total.

These variants form the candidate library fed into:
  → MMseqs2-GPU clustering (collapse near-identical sequences)
  → cuDF filtering (length, charge, hydrophobicity cutoffs)
  → AlphaFold3 complex prediction (top candidates vs. each target enzyme)

Inputs:
  propeptide_seeds.fasta   (output of extract_propeptides.py)

Outputs:
  variants_all.fasta       — all single-point variants + wild-type seeds
  variants_summary.tsv     — one row per variant: accession, position, wt_aa, mut_aa, sequence
"""

from pathlib import Path
from itertools import product

# ============================================================
# CONFIG
# ============================================================
SEEDS_FASTA   = "output/propeptides/propeptide_seeds.fasta"
OUT_DIR       = Path("output/variants")

# All 20 standard amino acids
ALL_AA = list("ACDEFGHIKLMNPQRSTVWY")

# Positions to skip (0-based).
# Leave empty to mutate every position.
# Fill in later once structural data (AlphaFold3) reveals critical interface residues
# — then you can protect those positions and restrict mutagenesis elsewhere.
SKIP_POSITIONS: dict[str, set[int]] = {
    # "P28784": {0, 1, 2},  # example: protect first 3 residues of RgpA propeptide
}

# Whether to include the wild-type seed sequences in the output
INCLUDE_WILDTYPE = True
# ============================================================


def parse_fasta(path: Path):
    """Yield (header, sequence) from a FASTA file."""
    header, seq_lines = None, []
    with open(path) as fh:
        for line in fh:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if header is not None:
                    yield header, "".join(seq_lines)
                header = line
                seq_lines = []
            else:
                seq_lines.append(line.strip())
    if header is not None:
        yield header, "".join(seq_lines)


def parse_seed_header(header: str) -> tuple[str, str, str]:
    """
    Parse '>P28784|propeptide|RgpA|25-227' into (accession, enzyme, coord_range).
    """
    parts = header.lstrip(">").split("|")
    acc        = parts[0] if len(parts) > 0 else "UNK"
    enzyme     = parts[2] if len(parts) > 2 else "UNK"
    coord      = parts[3] if len(parts) > 3 else "?"
    return acc, enzyme, coord


def generate_variants(acc: str, enzyme: str, coord: str, wt_seq: str):
    """
    Yield (variant_header, variant_seq) for every single-point substitution.
    Header format: >ACC|variant|WTpos+1MUT|enzyme|coord_range
    e.g.           >P28784|variant|Y25A|RgpA|25-227
    """
    skip = SKIP_POSITIONS.get(acc, set())
    for i, wt_aa in enumerate(wt_seq):
        if i in skip:
            continue
        pos_label = i + 1  # 1-based position within the propeptide fragment
        for mut_aa in ALL_AA:
            if mut_aa == wt_aa:
                continue  # skip synonymous (wild-type) substitution
            mut_seq = wt_seq[:i] + mut_aa + wt_seq[i+1:]
            header = f">{acc}|variant|{wt_aa}{pos_label}{mut_aa}|{enzyme}|{coord}"
            yield header, mut_seq


def main():
    seeds_path = Path(SEEDS_FASTA)
    if not seeds_path.exists():
        print(f"[error] Seeds file not found: {seeds_path}")
        print("        Run extract_propeptides.py first.")
        return

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    fasta_out   = OUT_DIR / "variants_all.fasta"
    summary_out = OUT_DIR / "variants_summary.tsv"

    seeds = list(parse_fasta(seeds_path))
    if not seeds:
        print("[error] No sequences found in seeds file.")
        return

    print(f"Loaded {len(seeds)} seed sequence(s) from {seeds_path.name}")

    total_variants = 0
    total_wt       = 0

    with open(fasta_out, "w") as fa, open(summary_out, "w") as tsv:
        tsv.write("accession\tenzyme\tcoord_range\tposition\twt_aa\tmut_aa\tsequence\n")

        for header, wt_seq in seeds:
            acc, enzyme, coord = parse_seed_header(header)
            skip = SKIP_POSITIONS.get(acc, set())

            print(f"\n  {acc} ({enzyme}): {len(wt_seq)} aa seed")
            print(f"    Positions to mutate: {len(wt_seq) - len(skip)} / {len(wt_seq)}")
            expected = (len(wt_seq) - len(skip)) * 19
            print(f"    Expected variants:   {expected}")

            # Write wild-type seed
            if INCLUDE_WILDTYPE:
                fa.write(f"{header}|wildtype\n{wt_seq}\n")
                tsv.write(f"{acc}\t{enzyme}\t{coord}\t0\tWT\tWT\t{wt_seq}\n")
                total_wt += 1

            # Write all single-point variants
            count = 0
            for var_header, var_seq in generate_variants(acc, enzyme, coord, wt_seq):
                fa.write(f"{var_header}\n{var_seq}\n")

                # Parse mutation label for TSV (e.g. "Y25A" -> pos=25, wt=Y, mut=A)
                mut_label = var_header.split("|")[2]      # e.g. "Y25A"
                wt_aa  = mut_label[0]
                mut_aa = mut_label[-1]
                pos    = mut_label[1:-1]
                tsv.write(f"{acc}\t{enzyme}\t{coord}\t{pos}\t{wt_aa}\t{mut_aa}\t{var_seq}\n")
                count += 1

            print(f"    Written:             {count} variants")
            total_variants += count

    print(f"\nDone.")
    print(f"  Wild-type seeds included: {total_wt}")
    print(f"  Total variants written:   {total_variants}")
    print(f"  FASTA output:             {fasta_out}")
    print(f"  TSV summary:              {summary_out}")
    print(f"\nNEXT STEPS:")
    print(f"  1. MMseqs2-GPU: cluster variants_all.fasta at 90% identity to collapse redundancy")
    print(f"     mmseqs easy-cluster variants_all.fasta clust_out tmp --min-seq-id 0.90 --gpu 1")
    print(f"  2. cuDF filtering: load variants_summary.tsv, apply length/charge/hydrophobicity cutoffs")
    print(f"  3. AlphaFold3: complex-predict top candidates paired against RgpA/RgpB/Kgp structures")


if __name__ == "__main__":
    main()

Loaded 3 seed sequence(s) from propeptide_seeds.fasta

  P95493 (RgpB): 205 aa seed
    Positions to mutate: 205 / 205
    Expected variants:   3895
    Written:             3895 variants

  P28784 (RgpA): 203 aa seed
    Positions to mutate: 203 / 203
    Expected variants:   3857
    Written:             3857 variants

  Q51817 (Kgp): 204 aa seed
    Positions to mutate: 204 / 204
    Expected variants:   3876
    Written:             3876 variants

Done.
  Wild-type seeds included: 3
  Total variants written:   11628
  FASTA output:             output/variants/variants_all.fasta
  TSV summary:              output/variants/variants_summary.tsv

NEXT STEPS:
  1. MMseqs2-GPU: cluster variants_all.fasta at 90% identity to collapse redundancy
     mmseqs easy-cluster variants_all.fasta clust_out tmp --min-seq-id 0.90 --gpu 1
  2. cuDF filtering: load variants_summary.tsv, apply length/charge/hydrophobicity cutoffs
  3. AlphaFold3: complex-predict top candidates paired against RgpA/RgpB/K

## Stage 2bii. Collecting Potential Propetides

Instead of generating potential gingipain propetides, collect potential candidates from existing database.

### Database 1: UNIPROT

In [1]:
#!/usr/bin/env python3
"""
collect_candidate_inhibitors.py
================================
Stage 1 (database track) of the gingipain inhibitor discovery pipeline.

This runs IN PARALLEL to propeptide_seeds.fasta -> generate_variants.py.
Instead of generating synthetic single-point mutants, this script pulls
NATURALLY OCCURRING / annotated peptide inhibitor candidates from UniProt,
so you end up with two independently-sourced candidate pools that both
funnel into the same AlphaFold3 scoring stage:

    variant track:   propeptide_seeds.fasta -> generate_variants.py -> cuDF filter -> AF3
    database track:  UniProt keyword search -> collect_candidate_inhibitors.py -> cuDF filter -> AF3

WHY UNIPROT AND NOT RAW MEROPS FILES
-------------------------------------
You already confirmed MEROPS C25.lib has zero curated propeptide entries
for gingipains specifically, and MEROPS's inhibitor library files (I*.lib)
are not published in a stable, parseable flat-file format. UniProt's REST
API is fully documented, returns clean JSON/FASTA, and lets us search by
annotated keyword + sequence length directly -- no manual file wrangling.

WHAT THIS COLLECTS
-------------------
UniProt entries carrying protease-inhibitor annotation keywords, restricted
to peptide-length sequences (default 10-80 aa) so they're AlphaFold3-complex
tractable. Two keyword sets are queried and merged:

  KW-0916  Protease inhibitor (broad, family/mechanism-agnostic)
  KW-0929  Cysteine protease inhibitor (mechanism-matched to gingipains,
           which are cysteine proteases despite clan CD/caspase-like fold
           rather than clan CA/papain-like)

Both sets are kept and tagged separately in the output metadata so you can
decide later whether to weight the mechanism-matched set more heavily.

OUTPUT
------
  output/candidates_db/candidates_raw.fasta   -- all collected sequences
  output/candidates_db/candidates_meta.tsv    -- accession, organism, length,
                                                  keyword_match, description
  output/candidates_db/collection_log.txt     -- run summary

These are RAW / unfiltered. The next stage (cuDF filter script, not yet
written) will apply length/charge/hydrophobicity/dedup-against-variant-pool
filtering before these become AlphaFold3 job candidates.
"""

import json
import re
import time
import urllib.request
import urllib.parse
from pathlib import Path
from datetime import datetime

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
CONFIG = {
    "output_dir": Path("output/candidates_db"),

    # UniProt REST search endpoint (no auth required)
    "base_url": "https://rest.uniprot.org/uniprotkb/search",

    # Peptide length window -- keep tractable for AF3 complex prediction
    "min_length": 10,
    "max_length": 80,

    # Keyword-based queries. Each is run separately so matches can be tagged
    # by which mechanism annotation they came from.
    "queries": [
        {
            "tag": "cysteine_protease_inhibitor",
            "keyword_id": "KW-0929",
        },
        {
            "tag": "protease_inhibitor_broad",
            "keyword_id": "KW-0916",
        },
    ],

    # Page size per UniProt request (max 500)
    "page_size": 500,

    # Politeness delay between paginated requests (seconds)
    "request_delay": 0.34,

    # Cap total entries pulled per query (raise once you've sanity-checked
    # the first run -- this keeps the first pass fast)
    "max_entries_per_query": 3000,
}


# ---------------------------------------------------------------------------
# UNIPROT QUERY
# ---------------------------------------------------------------------------
def build_query_url(keyword_id: str, min_len: int, max_len: int, cursor: str = None,
                     page_size: int = 500) -> str:
    """Build a UniProt REST search URL for a keyword + length-bounded peptide query."""
    query = f"(keyword:{keyword_id}) AND (length:[{min_len} TO {max_len}])"
    params = {
        "query": query,
        "format": "json",
        "fields": "accession,id,protein_name,organism_name,length,sequence,keyword",
        "size": str(page_size),
    }
    url = CONFIG["base_url"] + "?" + urllib.parse.urlencode(params)
    if cursor:
        url += f"&cursor={cursor}"
    return url


def fetch_page(url: str):
    """Fetch one page from UniProt REST API. Returns (json_body, next_cursor_or_None)."""
    req = urllib.request.Request(url, headers={"Accept": "application/json"})
    with urllib.request.urlopen(req, timeout=30) as resp:
        body = json.loads(resp.read().decode("utf-8"))
        link_header = resp.headers.get("Link", "")

    next_cursor = None
    if link_header:
        m = re.search(r'cursor=([^&>]+)', link_header)
        if m:
            next_cursor = m.group(1)
    return body, next_cursor


def collect_for_query(tag: str, keyword_id: str, min_len: int, max_len: int,
                       page_size: int, delay: float, max_entries: int, log_lines: list):
    """Paginate through UniProt results for one keyword query."""
    entries = []
    cursor = None
    url = build_query_url(keyword_id, min_len, max_len, cursor, page_size)

    while True:
        try:
            body, cursor = fetch_page(url)
        except Exception as e:
            log_lines.append(f"  [ERROR] {tag}: request failed ({e}); stopping this query.")
            break

        results = body.get("results", [])
        if not results:
            break

        for r in results:
            accession = r.get("primaryAccession", "")
            organism = r.get("organism", {}).get("scientificName", "unknown")
            length = r.get("sequence", {}).get("length", None)
            seq = r.get("sequence", {}).get("value", "")
            protein_desc = (
                r.get("proteinDescription", {})
                .get("recommendedName", {})
                .get("fullName", {})
                .get("value", "")
            )
            if not protein_desc:
                # fall back to submitted name if no recommended name
                submitted = r.get("proteinDescription", {}).get("submissionNames", [])
                if submitted:
                    protein_desc = submitted[0].get("fullName", {}).get("value", "")

            if not seq or not accession:
                continue

            entries.append({
                "accession": accession,
                "organism": organism,
                "length": length if length else len(seq),
                "sequence": seq,
                "description": protein_desc or "no description",
                "source_tag": tag,
            })

        log_lines.append(f"  {tag}: fetched {len(results)} entries this page "
                          f"(running total {len(entries)})")

        if len(entries) >= max_entries:
            log_lines.append(f"  {tag}: reached max_entries_per_query cap "
                              f"({max_entries}); stopping.")
            break

        if not cursor:
            break

        url = build_query_url(keyword_id, min_len, max_len, cursor, page_size)
        time.sleep(delay)

    return entries


# ---------------------------------------------------------------------------
# OUTPUT WRITERS
# ---------------------------------------------------------------------------
def write_fasta(entries: list, path: Path):
    with open(path, "w") as f:
        for e in entries:
            header = f">{e['accession']}|{e['source_tag']}|{e['organism'].replace(' ', '_')}"
            f.write(f"{header}\n{e['sequence']}\n")


def write_metadata_tsv(entries: list, path: Path):
    with open(path, "w") as f:
        f.write("accession\tsource_tag\torganism\tlength\tdescription\n")
        for e in entries:
            desc = e["description"].replace("\t", " ")
            f.write(f"{e['accession']}\t{e['source_tag']}\t{e['organism']}\t"
                    f"{e['length']}\t{desc}\n")


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------
def main():
    cfg = CONFIG
    out_dir = cfg["output_dir"]
    out_dir.mkdir(parents=True, exist_ok=True)

    log_lines = [f"collect_candidate_inhibitors.py run: {datetime.now().isoformat()}"]
    log_lines.append(f"Length window: {cfg['min_length']}-{cfg['max_length']} aa")

    all_entries = []
    seen_accessions = set()

    for q in cfg["queries"]:
        print(f"Querying UniProt: {q['tag']} ({q['keyword_id']}) ...")
        log_lines.append(f"\nQuery: {q['tag']} ({q['keyword_id']})")

        entries = collect_for_query(
            tag=q["tag"],
            keyword_id=q["keyword_id"],
            min_len=cfg["min_length"],
            max_len=cfg["max_length"],
            page_size=cfg["page_size"],
            delay=cfg["request_delay"],
            max_entries=cfg["max_entries_per_query"],
            log_lines=log_lines,
        )

        new_count = 0
        for e in entries:
            if e["accession"] not in seen_accessions:
                seen_accessions.add(e["accession"])
                all_entries.append(e)
                new_count += 1

        print(f"  -> {len(entries)} entries fetched, {new_count} new "
              f"(non-duplicate across queries)")
        log_lines.append(f"  -> {len(entries)} fetched, {new_count} new unique")

    # Write outputs
    fasta_path = out_dir / "candidates_raw.fasta"
    meta_path = out_dir / "candidates_meta.tsv"
    log_path = out_dir / "collection_log.txt"

    write_fasta(all_entries, fasta_path)
    write_metadata_tsv(all_entries, meta_path)

    log_lines.append(f"\nTOTAL unique candidates collected: {len(all_entries)}")
    log_lines.append(f"Written: {fasta_path}")
    log_lines.append(f"Written: {meta_path}")

    with open(log_path, "w") as f:
        f.write("\n".join(log_lines) + "\n")

    print(f"\nDone.")
    print(f"  Total unique candidates: {len(all_entries)}")
    print(f"  FASTA:    {fasta_path}")
    print(f"  Metadata: {meta_path}")
    print(f"  Log:      {log_path}")
    print(f"\nNEXT STEPS:")
    print(f"  1. Spot-check candidates_meta.tsv -- some 'protease inhibitor' hits")
    print(f"     will be broad-spectrum or serine/metallo-protease-specific, not")
    print(f"     cysteine-protease-specific. The cuDF filter stage should down-weight")
    print(f"     or flag entries only tagged 'protease_inhibitor_broad'.")
    print(f"  2. Feed candidates_raw.fasta into the cuDF filter script (length/charge/")
    print(f"     hydrophobicity + dedup against propeptide_all variant pool) once written.")
    print(f"  3. Tag surviving candidates with source='database' in the AF3 job manifest")
    print(f"     so parse_af3_results.py can compare ipTM/pTM by source (variant vs db).")


if __name__ == "__main__":
    main()

Querying UniProt: cysteine_protease_inhibitor (KW-0929) ...
  -> 3000 entries fetched, 3000 new (non-duplicate across queries)
Querying UniProt: protease_inhibitor_broad (KW-0916) ...
  -> 348 entries fetched, 348 new (non-duplicate across queries)

Done.
  Total unique candidates: 3348
  FASTA:    output/candidates_db/candidates_raw.fasta
  Metadata: output/candidates_db/candidates_meta.tsv
  Log:      output/candidates_db/collection_log.txt

NEXT STEPS:
  1. Spot-check candidates_meta.tsv -- some 'protease inhibitor' hits
     will be broad-spectrum or serine/metallo-protease-specific, not
     cysteine-protease-specific. The cuDF filter stage should down-weight
     or flag entries only tagged 'protease_inhibitor_broad'.
  2. Feed candidates_raw.fasta into the cuDF filter script (length/charge/
     hydrophobicity + dedup against propeptide_all variant pool) once written.
  3. Tag surviving candidates with source='database' in the AF3 job manifest
     so parse_af3_results.py can co

# Stage 3. Data Filtering

## Stage 3ai. BLOSUM62 Filtering (Comparative) for Generated Variants

In [4]:
#!/usr/bin/env python3
"""
filter_variants.py  (v3 — cross-enzyme conservation + per-enzyme reporting)
============================================================================
Filters single-point substitution variants using substitution-aware criteria.

Conservation fix (v2→v3):
  v2 compared strain variants of the SAME enzyme (P28784 vs B2RM93 for RgpA).
  RgpB and Kgp strain pairs were 204/205 and 204/204 identical — so the filter
  protected every position and rejected nearly all RgpB/Kgp candidates.

  v3 computes conservation ACROSS all three canonical seeds (RgpA vs RgpB vs Kgp).
  Positions identical in all three propeptides are structurally constrained
  (conserved across evolutionary distance, not just across strains), and are the
  ones actually worth protecting. Strain-variant comparison is discarded.

GPU: cupy for BLOSUM62 batch scoring. Falls back to numpy on CPU.

Inputs:
  output/variants/variants_summary.tsv
  output/variants/variants_all.fasta
  output/propeptides/propeptide_seeds.fasta   ← canonical seeds (cross-enzyme comparison)

Outputs:
  output/filtered/filtered_candidates.tsv
  output/filtered/filtered_candidates.fasta
  output/filtered/filter_report.txt
"""

from pathlib import Path
import numpy as np
import pandas as pd

try:
    import cupy as cp
    xp = cp
    print("[backend] cupy detected — BLOSUM62 scoring on GPU")
except ImportError:
    cp = None
    xp = np
    print("[backend] cupy not found — running on CPU (pip install cupy-cuda12x)")

# ============================================================
# CONFIG
# ============================================================
VARIANTS_TSV      = "output/variants/variants_summary.tsv"
VARIANTS_FASTA    = "output/variants/variants_all.fasta"
PROPEPTIDE_SEEDS  = "output/propeptides/propeptide_seeds.fasta"   # 3 canonical seeds
OUT_DIR           = Path("output/filtered")

FILTERS = {
    # BLOSUM62 score threshold. -1 accepts mildly disfavoured swaps.
    # Raise to 0 to keep only neutral-or-better substitutions.
    "blosum62_min":          -1,

    # Reject substitutions that introduce cysteine where none existed.
    "no_new_cysteine":       True,

    # Reject substitutions TO proline at non-proline positions.
    "no_proline_injection":  True,

    # Protect positions conserved across ALL THREE canonical propeptides
    # (RgpA vs RgpB vs Kgp). These are structurally constrained positions —
    # conserved across evolutionary distance, not just across near-identical strains.
    "protect_conserved":     True,
}
# ============================================================


# ── BLOSUM62 ─────────────────────────────────────────────────────────────────
AA_ORDER = "ARNDCQEGHILKMFPSTWYV"
AA_INDEX = {aa: i for i, aa in enumerate(AA_ORDER)}

# fmt: off
_BLOSUM62 = [
#    A   R   N   D   C   Q   E   G   H   I   L   K   M   F   P   S   T   W   Y   V
    [4, -1, -2, -2,  0, -1, -1,  0, -2, -1, -1, -1, -1, -2, -1,  1,  0, -3, -2,  0],
   [-1,  5,  0, -2, -3,  1,  0, -2,  0, -3, -2,  2, -1, -3, -2, -1, -1, -3, -2, -3],
   [-2,  0,  6,  1, -3,  0,  0,  0,  1, -3, -3,  0, -2, -3, -2,  1,  0, -4, -2, -3],
   [-2, -2,  1,  6, -3,  0,  2, -1, -1, -3, -4, -1, -3, -3, -1,  0, -1, -4, -3, -3],
   [ 0, -3, -3, -3,  9, -3, -4, -3, -3, -1, -1, -3, -1, -2, -3, -1, -1, -2, -2, -1],
   [-1,  1,  0,  0, -3,  5,  2, -2,  0, -3, -2,  1,  0, -3, -1,  0, -1, -2, -1, -2],
   [-1,  0,  0,  2, -4,  2,  5, -2,  0, -3, -3,  1, -2, -3, -1,  0, -1, -3, -2, -2],
   [ 0, -2,  0, -1, -3, -2, -2,  6, -2, -4, -4, -2, -3, -3, -2,  0, -2, -2, -3, -3],
   [-2,  0,  1, -1, -3,  0,  0, -2,  8, -3, -3, -1, -2, -1, -2, -1, -2, -2,  2, -3],
   [-1, -3, -3, -3, -1, -3, -3, -4, -3,  4,  2, -3,  1,  0, -3, -2, -1, -3, -1,  3],
   [-1, -2, -3, -4, -1, -2, -3, -4, -3,  2,  4, -2,  2,  0, -3, -2, -1, -2, -1,  1],
   [-1,  2,  0, -1, -3,  1,  1, -2, -1, -3, -2,  5, -1, -3, -1,  0, -1, -3, -2, -2],
   [-1, -1, -2, -3, -1,  0, -2, -3, -2,  1,  2, -1,  5,  0, -2, -1, -1, -1, -1,  1],
   [-2, -3, -3, -3, -2, -3, -3, -3, -1,  0,  0, -3,  0,  6, -4, -2, -2,  1,  3, -1],
   [-1, -2, -2, -1, -3, -1, -1, -2, -2, -3, -3, -1, -2, -4,  7, -1, -1, -4, -3, -2],
   [ 1, -1,  1,  0, -1,  0,  0,  0, -1, -2, -2,  0, -1, -2, -1,  4,  1, -3, -2, -2],
   [ 0, -1,  0, -1, -1, -1, -1, -2, -2, -1, -1, -1, -1, -2, -1,  1,  5, -2, -2,  0],
   [-3, -3, -4, -4, -2, -2, -3, -2, -2, -3, -2, -3, -1,  1, -4, -3, -2, 11,  2, -3],
   [-2, -2, -2, -3, -2, -1, -2, -3,  2, -1, -1, -2, -1,  3, -3, -2, -2,  2,  7, -1],
   [ 0, -3, -3, -3, -1, -2, -2, -3, -3,  3,  1, -2,  1, -1, -2, -2,  0, -3, -1,  4],
]
# fmt: on

BLOSUM62_NP = np.array(_BLOSUM62, dtype=np.int8)
BLOSUM62_XP = xp.array(_BLOSUM62, dtype=xp.int8)


def blosum62_batch(wt_aas: list, mut_aas: list) -> np.ndarray:
    i = np.array([AA_INDEX.get(a, 0) for a in wt_aas], dtype=np.int32)
    j = np.array([AA_INDEX.get(a, 0) for a in mut_aas], dtype=np.int32)
    if cp is not None:
        scores = BLOSUM62_XP[cp.asarray(i), cp.asarray(j)]
        return cp.asnumpy(scores).astype(np.int32)
    return BLOSUM62_NP[i, j].astype(np.int32)


# ── FASTA helpers ─────────────────────────────────────────────────────────────

def parse_fasta(path: Path):
    header, seq_lines = None, []
    with open(path) as fh:
        for line in fh:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if header is not None:
                    yield header, "".join(seq_lines)
                header = line
                seq_lines = []
            else:
                seq_lines.append(line.strip())
    if header is not None:
        yield header, "".join(seq_lines)


def index_fasta(path: Path) -> dict:
    idx = {}
    for header, seq in parse_fasta(path):
        parts = header.lstrip(">").split("|")
        acc = parts[0]
        mut = parts[2] if len(parts) > 2 else "WT"
        idx[f"{acc}_{mut}"] = (header, seq)
    return idx


# ── Cross-enzyme conservation ─────────────────────────────────────────────────

def find_cross_enzyme_conserved(seeds_path: Path) -> set:
    """
    Load all canonical seed propeptides and find positions where every seed
    has the same amino acid (column-wise, truncated to shortest sequence).

    This is cross-enzyme conservation (RgpA vs RgpB vs Kgp), which reflects
    genuine structural constraint — not strain-level identity (~99%), which
    is uninformative.

    Returns a set of 0-based positions conserved across all seeds.
    """
    seeds = []
    labels = []
    for header, seq in parse_fasta(seeds_path):
        parts = header.lstrip(">").split("|")
        acc   = parts[0]
        label = parts[2] if len(parts) > 2 else acc
        seeds.append(seq)
        labels.append(f"{label} ({acc}, {len(seq)} aa)")

    if len(seeds) < 2:
        print("  Only 1 seed found — skipping cross-enzyme conservation")
        return set()

    min_len = min(len(s) for s in seeds)
    print(f"  Seeds: {', '.join(labels)}")
    print(f"  Comparing {len(seeds)} seeds across {min_len} positions (truncated to shortest)")

    conserved = set()
    for pos in range(min_len):
        residues = {s[pos] for s in seeds}
        if len(residues) == 1:
            conserved.add(pos)

    pct = 100 * len(conserved) / min_len
    print(f"  Cross-enzyme conserved: {len(conserved)}/{min_len} positions ({pct:.1f}%)")
    print(f"  These positions are protected from mutagenesis.")
    return conserved


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    for p in [Path(VARIANTS_TSV), Path(VARIANTS_FASTA), Path(PROPEPTIDE_SEEDS)]:
        if not p.exists():
            print(f"[error] File not found: {p}")
            return

    OUT_DIR.mkdir(parents=True, exist_ok=True)

    # 1. Load variants
    print(f"Loading {VARIANTS_TSV} ...")
    df = pd.read_csv(VARIANTS_TSV, sep="\t")
    df = df[df["mut_aa"] != "WT"].copy()
    print(f"  {len(df)} variant rows across {df['accession'].nunique()} enzymes\n")

    # 2. Cross-enzyme conserved positions
    conserved_positions = set()
    if FILTERS["protect_conserved"]:
        print("Computing cross-enzyme conserved positions ...")
        conserved_positions = find_cross_enzyme_conserved(Path(PROPEPTIDE_SEEDS))
        print()

    # 3. Compute substitution features
    print("Computing substitution features ...")
    df["blosum62_score"]      = blosum62_batch(df["wt_aa"].tolist(), df["mut_aa"].tolist())
    df["introduces_cysteine"] = (df["wt_aa"] != "C") & (df["mut_aa"] == "C")
    df["injects_proline"]     = (df["wt_aa"] != "P") & (df["mut_aa"] == "P")
    df["at_conserved_pos"]    = df["position"].apply(
        lambda p: (int(p) - 1) in conserved_positions  # position is 1-based in TSV
    )

    # 4. Build mask
    mask = pd.Series(True, index=df.index)
    filter_specs = []

    b_mask = df["blosum62_score"] >= FILTERS["blosum62_min"]
    filter_specs.append(("BLOSUM62 ≥ min", b_mask))
    mask &= b_mask

    if FILTERS["no_new_cysteine"]:
        c_mask = ~df["introduces_cysteine"]
        filter_specs.append(("No new cysteine", c_mask))
        mask &= c_mask

    if FILTERS["no_proline_injection"]:
        p_mask = ~df["injects_proline"]
        filter_specs.append(("No proline injection", p_mask))
        mask &= p_mask

    if FILTERS["protect_conserved"] and conserved_positions:
        cons_mask = ~df["at_conserved_pos"]
        filter_specs.append(("Not at cross-enzyme conserved pos", cons_mask))
        mask &= cons_mask

    df_filtered = df[mask].copy()

    # 5. Report
    sep = "=" * 62
    print(f"\nFILTER REPORT")
    print(sep)
    print(f"  {'Input variants:':<36} {len(df):>5}")
    for name, filt_mask in filter_specs:
        n_fail = int((~filt_mask).sum())
        pct    = 100 * n_fail / len(df)
        print(f"  {name:<36} {n_fail:>5} failed  ({pct:.1f}%)")
    print("-" * 62)
    n_pass = len(df_filtered)
    print(f"  {'Passed all filters:':<36} {n_pass:>5} / {len(df)}")
    print(f"  {'Rejection rate:':<36} {100*(1-n_pass/len(df)):.1f}%")

    # Per-enzyme breakdown
    print(f"\n  Per-enzyme breakdown (passing variants):")
    per_enzyme = df_filtered.groupby("accession").size()
    for acc, count in per_enzyme.items():
        enzyme = df_filtered[df_filtered["accession"] == acc]["enzyme"].iloc[0]
        total  = int((df["accession"] == acc).sum())
        print(f"    {acc} ({enzyme:<6}): {count:>4} / {total}  ({100*count/total:.1f}%)")

    # BLOSUM62 distribution
    print(f"\n  BLOSUM62 score distribution:")
    for score in sorted(df["blosum62_score"].unique()):
        n   = int((df["blosum62_score"] == score).sum())
        bar = "█" * (n // 50)
        print(f"    {score:>3}: {n:>5}  {bar}")

    print(sep)

    # 6. Write outputs
    tsv_out = OUT_DIR / "filtered_candidates.tsv"
    df_filtered.to_csv(tsv_out, sep="\t", index=False)
    print(f"\nTSV:    {tsv_out}")

    report_lines = [
        f"Input: {len(df)}",
        f"Passed: {n_pass}",
        f"Rejection rate: {100*(1-n_pass/len(df)):.1f}%",
        f"Cross-enzyme conserved positions protected: {len(conserved_positions)}",
    ]
    for acc, count in per_enzyme.items():
        report_lines.append(f"  {acc}: {count} passing")
    (OUT_DIR / "filter_report.txt").write_text("\n".join(report_lines) + "\n")
    print(f"Report: {OUT_DIR / 'filter_report.txt'}")

    print(f"Indexing variants FASTA ...")
    fasta_idx = index_fasta(Path(VARIANTS_FASTA))
    fasta_out = OUT_DIR / "filtered_candidates.fasta"
    found, missing = 0, 0
    with open(fasta_out, "w") as fa:
        for _, row in df_filtered.iterrows():
            key = f"{row['accession']}_{row['wt_aa']}{row['position']}{row['mut_aa']}"
            if key in fasta_idx:
                header, seq = fasta_idx[key]
                fa.write(f"{header}\n{seq}\n")
                found += 1
            else:
                missing += 1
    print(f"FASTA:  {fasta_out}  ({found} written, {missing} missing)")

    print(f"\nDone.")
    print(f"\nNEXT STEP — position-based clustering (NOT MMseqs2 identity clustering):")
    print(f"  Single-point mutants are 99.5% identical — MMseqs2 at 90% would collapse")
    print(f"  all variants to 3 representatives (one per enzyme). Instead, run:")
    print(f"  python cluster_by_position.py")


if __name__ == "__main__":
    main()

[backend] cupy detected — BLOSUM62 scoring on GPU
Loading output/variants/variants_summary.tsv ...
  11628 variant rows across 3 enzymes

Computing cross-enzyme conserved positions ...
  Seeds: RgpB (P95493, 205 aa), RgpA (P28784, 203 aa), Kgp (Q51817, 204 aa)
  Comparing 3 seeds across 203 positions (truncated to shortest)
  Cross-enzyme conserved: 11/203 positions (5.4%)
  These positions are protected from mutagenesis.

Computing substitution features ...

FILTER REPORT
  Input variants:                      11628
  BLOSUM62 ≥ min                        5726 failed  (49.2%)
  No new cysteine                        611 failed  (5.3%)
  No proline injection                   574 failed  (4.9%)
  Not at cross-enzyme conserved pos      627 failed  (5.4%)
--------------------------------------------------------------
  Passed all filters:                   5080 / 11628
  Rejection rate:                      56.3%

  Per-enzyme breakdown (passing variants):
    P28784 (RgpA  ): 1684 / 385

## Stage 3aii. cuDF Biophysical Filter (Non-Comparative) for Database Candidates

In [4]:
#!/usr/bin/env python3
"""
filter_candidates_gpu.py
=========================
Stage 2 (database track) of the gingipain inhibitor discovery pipeline.

Input:  output/candidates_db/candidates_raw.fasta + candidates_meta.tsv
        (from collect_candidate_inhibitors.py)
Output: output/candidates_db/candidates_filtered.fasta + candidates_filtered_meta.tsv

ENV NOTE
--------
Rewritten for real RAPIDS cuDF now that you're on Python 3.11 (RAPIDS supports
3.10-3.12). The dataframe layer (load, merge, filter, sort, write) is now
actual cudf.DataFrame, not pandas. Per-residue property computation still
uses cupy directly (charge/GRAVY/MW are array reductions, not naturally
dataframe-shaped operations) -- cudf and cupy interop natively, a cupy array
assigns straight into a cudf column with no host round-trip.

WHY BULK BIOPHYSICAL FILTERS WORK HERE (UNLIKE THE VARIANT TRACK)
-------------------------------------------------------------------
Your v1 variant filter using GRAVY/MW/charge produced 0% rejection because
single-point mutants are near-identical at the whole-sequence level -- one
substitution barely moves bulk properties. That's NOT true here: these are
independently-sourced natural peptides with real diversity in length,
charge, and hydrophobicity, so bulk filtering is the correct tool for this
track (substitution-level filtering, by contrast, would be meaningless here
since there's no shared wild-type scaffold to substitute against).

FILTER LOGIC
------------
HARD REJECTS (removed from output):
  - non-standard residues (X, B, Z, U, O, J) -- AlphaFold3 needs canonical AAs
  - length outside [8, 80] aa (safety recheck; collection already bounded this)
  - |net charge| > 10 -- very likely disordered / membrane-lytic AMP rather
    than a folded competitive active-site inhibitor
  - GRAVY > 2.5 -- extreme hydrophobicity, high aggregation risk
  - exact sequence duplicate of anything already in your variant-track pool
    (propeptide_all.fasta) -- no point spending an AF3 job on it twice

SOFT SCORING (kept, but ranked -- does not reject):
  - source_tag: cysteine_protease_inhibitor entries score higher than
    protease_inhibitor_broad, since the latter includes serine/metallo
    inhibitors with no real mechanistic reason to bind a cysteine protease
    active site
  - charge proximity to neutral (propeptide active-site-blocking loops in
    the natural mechanism are not strongly charged)
  - length in the 10-30 aa "typical peptide inhibitor" sweet spot

NOTE ON NEAR-DUPLICATE CLUSTERING
-----------------------------------
This script only removes EXACT duplicates against the variant pool. Collapsing
near-identical natural sequences (>90% identity) is exactly the job you already
identified for MMseqs2-GPU, which is appropriate for this track (unlike the
variant track, where 99.5% pairwise identity would collapse everything). Run
MMseqs2 clustering as the step AFTER this filter, not before.
"""

import re
from pathlib import Path
from datetime import datetime

import numpy as np
import cupy as cp
import cudf

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
CONFIG = {
    "candidates_dir": Path("output/candidates_db"),
    "raw_fasta": Path("output/candidates_db/candidates_raw.fasta"),
    "meta_tsv": Path("output/candidates_db/candidates_meta.tsv"),

    # Variant-track pool to dedup against (canonical + strain propeptides;
    # point to the broader all-variants file, not just the 3 seeds)
    "variant_pool_fasta": Path("output/propeptides/propeptide_all.fasta"),

    # Hard-reject thresholds
    "min_length": 8,
    "max_length": 80,
    "max_abs_charge": 10.0,
    "max_gravy": 2.5,

    # Soft-scoring weights
    "source_tag_weights": {
        "cysteine_protease_inhibitor": 2.0,
        "protease_inhibitor_broad": 0.0,
    },
    "sweet_spot_len_min": 10,
    "sweet_spot_len_max": 30,
    "sweet_spot_bonus": 1.0,
    "neutral_charge_bonus_max": 1.0,   # bonus scales down as |charge| grows
}

# Kyte-Doolittle hydropathy index
KD_HYDROPATHY = {
    "A": 1.8, "R": -4.5, "N": -3.5, "D": -3.5, "C": 2.5,
    "Q": -3.5, "E": -3.5, "G": -0.4, "H": -3.2, "I": 4.5,
    "L": 3.8, "K": -3.9, "M": 1.9, "F": 2.8, "P": -1.6,
    "S": -0.8, "T": -0.7, "W": -0.9, "Y": -1.3, "V": 4.2,
}

# Average residue masses (monoisotopic-ish average, Da), + 18.02 for water
RESIDUE_MASS = {
    "A": 71.08, "R": 156.19, "N": 114.10, "D": 115.09, "C": 103.14,
    "Q": 128.13, "E": 129.12, "G": 57.05, "H": 137.14, "I": 113.16,
    "L": 113.16, "K": 128.17, "M": 131.19, "F": 147.18, "P": 97.12,
    "S": 87.08, "T": 101.10, "W": 186.21, "Y": 163.18, "V": 99.13,
}

STANDARD_AA = set(KD_HYDROPATHY.keys())
NON_STANDARD_PATTERN = re.compile(r"[^ACDEFGHIKLMNPQRSTVWY]")


# ---------------------------------------------------------------------------
# FASTA I/O
# ---------------------------------------------------------------------------
def read_fasta(path: Path):
    """Minimal FASTA reader -> list of (header, sequence)."""
    entries = []
    header = None
    seq_chunks = []
    with open(path) as f:
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    entries.append((header, "".join(seq_chunks)))
                header = line[1:]
                seq_chunks = []
            else:
                seq_chunks.append(line.strip())
    if header is not None:
        entries.append((header, "".join(seq_chunks)))
    return entries


def write_fasta(rows: "cudf.DataFrame", path: Path):
    # iterrows-style writing needs host memory; to_pandas() is a small,
    # one-time transfer of the already-filtered (small) result set.
    host_rows = rows.to_pandas()
    with open(path, "w") as f:
        for _, r in host_rows.iterrows():
            f.write(f">{r['accession']}|{r['source_tag']}|score={r['priority_score']:.2f}\n")
            f.write(f"{r['sequence']}\n")


# ---------------------------------------------------------------------------
# GPU-VECTORIZED PROPERTY CALCULATION
# ---------------------------------------------------------------------------
def compute_properties_gpu(sequences: list[str]):
    """
    Compute length, net charge, GRAVY, and MW for a list of sequences using
    cupy. Approach: build per-residue lookup arrays, then reduce on GPU.
    Sequences are padded to max length; padding contributes zero via masks.
    """
    n = len(sequences)
    max_len = max(len(s) for s in sequences) if n else 0

    aa_list = sorted(STANDARD_AA)
    aa_index = {aa: i for i, aa in enumerate(aa_list)}

    hydropathy_table = cp.array([KD_HYDROPATHY[aa] for aa in aa_list], dtype=cp.float32)
    mass_table = cp.array([RESIDUE_MASS[aa] for aa in aa_list], dtype=cp.float32)
    # charge contribution per residue at ~pH 7 (simplified Henderson-Hasselbalch-free model)
    charge_table = cp.array(
        [1.0 if aa in ("K", "R") else 0.1 if aa == "H" else -1.0 if aa in ("D", "E") else 0.0
         for aa in aa_list],
        dtype=cp.float32,
    )

    # Build (n, max_len) index matrix, -1 for padding/non-standard
    idx_matrix = np.full((n, max_len), -1, dtype=np.int32)
    lengths = np.zeros(n, dtype=np.int32)
    for i, seq in enumerate(sequences):
        lengths[i] = len(seq)
        for j, aa in enumerate(seq):
            idx_matrix[i, j] = aa_index.get(aa, -1)

    idx_gpu = cp.array(idx_matrix)
    valid_mask = (idx_gpu >= 0)
    safe_idx = cp.where(valid_mask, idx_gpu, 0)

    hydro_vals = cp.where(valid_mask, hydropathy_table[safe_idx], 0.0)
    mass_vals = cp.where(valid_mask, mass_table[safe_idx], 0.0)
    charge_vals = cp.where(valid_mask, charge_table[safe_idx], 0.0)

    lengths_gpu = cp.array(lengths, dtype=cp.float32)
    lengths_safe = cp.where(lengths_gpu > 0, lengths_gpu, 1.0)

    gravy = cp.sum(hydro_vals, axis=1) / lengths_safe
    net_charge = cp.sum(charge_vals, axis=1)
    mw = cp.sum(mass_vals, axis=1) + 18.02  # + water

    return {
        "gravy": cp.asnumpy(gravy),
        "net_charge": cp.asnumpy(net_charge),
        "mw": cp.asnumpy(mw),
        "length": lengths,
    }


# ---------------------------------------------------------------------------
# FILTER + SCORE
# ---------------------------------------------------------------------------
def apply_filters(df: "cudf.DataFrame", cfg: dict, variant_seqs: set, log_lines: list):
    n_start = len(df)
    log_lines.append(f"Starting candidates: {n_start}")

    # 1. Non-standard residues.
    # cuDF's .str accessor supports regex containment natively (GPU-resident,
    # no Python-level .apply -- .apply on cuDF rows falls back to slow
    # row-by-row UDF compilation and defeats the point of being on GPU).
    has_nonstandard = df["sequence"].str.contains(NON_STANDARD_PATTERN.pattern, regex=True)
    n_nonstandard = int(has_nonstandard.sum())
    df = df[~has_nonstandard].reset_index(drop=True)
    log_lines.append(f"  Rejected (non-standard residues): {n_nonstandard}")

    # 2. Length window
    too_short = df["length"] < cfg["min_length"]
    too_long = df["length"] > cfg["max_length"]
    log_lines.append(f"  Rejected (length < {cfg['min_length']}): {int(too_short.sum())}")
    log_lines.append(f"  Rejected (length > {cfg['max_length']}): {int(too_long.sum())}")
    df = df[~(too_short | too_long)].reset_index(drop=True)

    # 3. Exact duplicate of variant-track pool.
    # isin against a Python set works in cuDF the same way as pandas.
    is_dup = df["sequence"].isin(variant_seqs)
    log_lines.append(f"  Rejected (exact duplicate of variant pool): {int(is_dup.sum())}")
    df = df[~is_dup].reset_index(drop=True)

    # 4. Recompute properties on the surviving set (cupy array -> cudf column,
    # zero-copy device-to-device assignment)
    if len(df) == 0:
        log_lines.append("  No candidates survived hard filters -- stopping.")
        return df

    props = compute_properties_gpu(df["sequence"].to_pandas().tolist())
    df["net_charge"] = cp.asarray(props["net_charge"])
    df["gravy"] = cp.asarray(props["gravy"])
    df["mw"] = cp.asarray(props["mw"])

    # 5. Charge / hydrophobicity extremes
    extreme_charge = df["net_charge"].abs() > cfg["max_abs_charge"]
    extreme_gravy = df["gravy"] > cfg["max_gravy"]
    log_lines.append(f"  Rejected (|net charge| > {cfg['max_abs_charge']}): {int(extreme_charge.sum())}")
    log_lines.append(f"  Rejected (GRAVY > {cfg['max_gravy']}, aggregation risk): {int(extreme_gravy.sum())}")
    df = df[~(extreme_charge | extreme_gravy)].reset_index(drop=True)

    log_lines.append(f"Survived hard filters: {len(df)} / {n_start} "
                      f"({100 * len(df) / n_start:.1f}%)")

    if len(df) == 0:
        return df

    # 6. Soft priority score (does not reject, only ranks).
    # cuDF Series.map works with a plain dict same as pandas.
    tag_weight = df["source_tag"].map(cfg["source_tag_weights"]).fillna(0.0)
    sweet_spot = ((df["length"] >= cfg["sweet_spot_len_min"]) &
                  (df["length"] <= cfg["sweet_spot_len_max"])).astype("float32") * cfg["sweet_spot_bonus"]
    # neutral-charge bonus decays linearly to 0 at max_abs_charge
    neutral_bonus = (1 - df["net_charge"].abs() / cfg["max_abs_charge"]).clip(lower=0) * cfg["neutral_charge_bonus_max"]

    df["priority_score"] = tag_weight + sweet_spot + neutral_bonus
    df = df.sort_values("priority_score", ascending=False).reset_index(drop=True)

    return df


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------
def main():
    cfg = CONFIG
    log_lines = [f"filter_candidates_gpu.py run: {datetime.now().isoformat()}"]

    # Load raw candidates
    raw_entries = read_fasta(cfg["raw_fasta"])
    print(f"Loaded {len(raw_entries)} raw candidate sequences from {cfg['raw_fasta']}")

    rows = []
    for header, seq in raw_entries:
        parts = header.split("|")
        accession = parts[0] if len(parts) > 0 else header
        source_tag = parts[1] if len(parts) > 1 else "unknown"
        rows.append({"accession": accession, "source_tag": source_tag,
                      "sequence": seq, "length": len(seq)})
    df = cudf.DataFrame(rows)

    # Merge in description/organism from meta tsv if present.
    # cudf.read_csv reads straight to GPU (skips the pandas host round-trip
    # you'd get from pd.read_csv + a later cudf.from_pandas conversion).
    if cfg["meta_tsv"].exists():
        meta = cudf.read_csv(cfg["meta_tsv"], sep="\t")
        df = df.merge(meta[["accession", "organism", "description"]],
                       on="accession", how="left")
    else:
        log_lines.append(f"WARNING: {cfg['meta_tsv']} not found -- proceeding without "
                          f"organism/description columns.")

    # Load variant-track pool for dedup
    variant_seqs = set()
    if cfg["variant_pool_fasta"].exists():
        variant_entries = read_fasta(cfg["variant_pool_fasta"])
        variant_seqs = {seq for _, seq in variant_entries}
        print(f"Loaded {len(variant_seqs)} variant-track sequences for dedup "
              f"from {cfg['variant_pool_fasta']}")
    else:
        log_lines.append(f"WARNING: variant pool {cfg['variant_pool_fasta']} not found -- "
                          f"skipping dedup-against-variants step.")
        print(f"WARNING: {cfg['variant_pool_fasta']} not found; skipping variant dedup.")

    filtered = apply_filters(df, cfg, variant_seqs, log_lines)

    out_dir = cfg["candidates_dir"]
    out_dir.mkdir(parents=True, exist_ok=True)
    fasta_out = out_dir / "candidates_filtered.fasta"
    meta_out = out_dir / "candidates_filtered_meta.tsv"
    log_out = out_dir / "filter_log.txt"

    if len(filtered) > 0:
        write_fasta(filtered, fasta_out)
        cols = ["accession", "source_tag", "organism", "description", "length",
                "net_charge", "gravy", "mw", "priority_score"]
        cols = [c for c in cols if c in filtered.columns]
        filtered[cols].to_csv(meta_out, sep="\t", index=False)  # cudf.to_csv, same call shape as pandas

    with open(log_out, "w") as f:
        f.write("\n".join(log_lines) + "\n")

    print("\n".join(log_lines))
    print(f"\nDone.")
    print(f"  Filtered candidates: {len(filtered)}")
    print(f"  FASTA:    {fasta_out}")
    print(f"  Metadata: {meta_out}")
    print(f"  Log:      {log_out}")
    print(f"\nNEXT STEPS:")
    print(f"  1. Sanity-check the top of candidates_filtered_meta.tsv (sorted by")
    print(f"     priority_score) -- cysteine_protease_inhibitor entries near the")
    print(f"     10-30 aa sweet spot with near-neutral charge should be ranked first.")
    print(f"  2. Run MMseqs2-GPU clustering on candidates_filtered.fasta to collapse")
    print(f"     near-identical natural sequences (this is where MMseqs2 belongs in")
    print(f"     this track -- unlike the variant track).")
    print(f"  3. Feed clustered representatives into AF3 job generation, tagged")
    print(f"     source='database' in the job manifest, alongside the variant-track")
    print(f"     candidates tagged source='variant'.")


if __name__ == "__main__":
    main()

Loaded 3348 raw candidate sequences from output/candidates_db/candidates_raw.fasta
Loaded 5 variant-track sequences for dedup from output/propeptides/propeptide_all.fasta
filter_candidates_gpu.py run: 2026-07-06T11:59:24.939734
Starting candidates: 3348
  Rejected (non-standard residues): 54
  Rejected (length < 8): 0
  Rejected (length > 80): 0
  Rejected (exact duplicate of variant pool): 0
  Rejected (|net charge| > 10.0): 121
  Rejected (GRAVY > 2.5, aggregation risk): 0
Survived hard filters: 3173 / 3348 (94.8%)

Done.
  Filtered candidates: 3173
  FASTA:    output/candidates_db/candidates_filtered.fasta
  Metadata: output/candidates_db/candidates_filtered_meta.tsv
  Log:      output/candidates_db/filter_log.txt

NEXT STEPS:
  1. Sanity-check the top of candidates_filtered_meta.tsv (sorted by
     priority_score) -- cysteine_protease_inhibitor entries near the
     10-30 aa sweet spot with near-neutral charge should be ranked first.
  2. Run MMseqs2-GPU clustering on candidates_fi

## Stage 3bi. Top-N Clustering

For a single-point mutant library the right clustering strategy is by position, not by identity. Group all mutations at the same position together and pick the representative with the highest BLOSUM62 score. That gives you at most ~L unique position clusters per enzyme, each represented by the most plausible substitution — a natural reduction of candidates without losing positional diversity.

Top-1 per position picks the single best-scoring substitution at each site. For a position where the wild-type is Alanine, you'd get exactly one candidate — say A→V (score +1, most conservative hydrophobic swap). You end up with at most 609 sequences total (203 positions × 3 enzymes). Clean and minimal, but you might miss a useful substitution at the same position with a slightly lower score that has a very different chemical character.

Top-N per position keeps the N best-scoring substitutions at each site. For the same Alanine position you might keep A→V (+1), A→I (+1), A→L (+0) — three candidates covering similar hydrophobic swaps. At N=3 you'd get up to 1,827 sequences. The tradeoff is more AlphaFold3 jobs, but you get chemical diversity within each position, which matters because BLOSUM62 score measures evolutionary plausibility, not binding affinity — a score-2 substitution might still bind better than score-1 if it adds a key hydrogen bond.
The practical middle ground for this pipeline: top-2 per position, but only if the two substitutions differ in chemical class (e.g. don't keep both A→V and A→I since they're both small hydrophobics — but do keep A→V and A→K since one is hydrophobic and one is charged). That way you get chemical diversity without redundancy.

In [ ]:
#!/usr/bin/env python3
"""
cluster_by_position.py
======================
Reduces filtered propeptide variants by clustering on mutation position,
keeping the top-N substitutions per (enzyme, position) pair.

Why not MMseqs2 identity clustering?
  Single-point mutants of a ~204 aa sequence are 99.5% identical to each other.
  MMseqs2 at 90% identity would collapse all variants into 3 representatives
  (one per enzyme), discarding all positional diversity. Position-based
  clustering is the correct approach for single-point mutant libraries.

Clustering strategy:
  For each (enzyme, position) group:
    1. Rank candidates by BLOSUM62 score (descending)
    2. Apply chemical class diversity filter if TOP_N > 1:
       only add a candidate if it belongs to a chemical class not yet
       represented in the selected set for that position
    3. Keep up to TOP_N representatives

Chemical classes (7 groups):
  nonpolar_aliphatic : G A V L I M P
  aromatic           : F W Y
  polar_uncharged    : S T C N Q
  positive           : K R H
  negative           : D E
  special_cys        : C  (split out due to disulfide concerns — already filtered)
  special_pro        : P  (split out due to structural rigidity — already filtered)

Inputs:
  output/filtered/filtered_candidates.tsv
  output/filtered/filtered_candidates.fasta

Outputs:
  output/clustered/clustered_representatives.tsv
  output/clustered/clustered_representatives.fasta
  output/clustered/clustering_report.txt
"""

from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np

# ============================================================
# CONFIG
# ============================================================
FILTERED_TSV   = "output/filtered/filtered_candidates.tsv"
FILTERED_FASTA = "output/filtered/filtered_candidates.fasta"
OUT_DIR        = Path("output/topn_clustered")

# Number of substitutions to keep per (enzyme, position).
# TOP_N = 1 : minimal library, best BLOSUM62 score only   → ≤ 609 sequences
# TOP_N = 2 : recommended, adds chemical class diversity  → ≤ 1218 sequences
# TOP_N = 3 : broader coverage                           → ≤ 1827 sequences
TOP_N = 2

# If True and TOP_N > 1, enforce that each kept substitution at a position
# must introduce a residue from a different chemical class than those already
# selected. Prevents keeping e.g. both A→V and A→I (both nonpolar aliphatic).
CHEMICAL_DIVERSITY = True
# ============================================================


# ── Chemical class definitions ────────────────────────────────────────────────
CHEMICAL_CLASS = {
    "G": "nonpolar_aliphatic", "A": "nonpolar_aliphatic",
    "V": "nonpolar_aliphatic", "L": "nonpolar_aliphatic",
    "I": "nonpolar_aliphatic", "M": "nonpolar_aliphatic",
    "P": "special_pro",
    "F": "aromatic",           "W": "aromatic",           "Y": "aromatic",
    "S": "polar_uncharged",    "T": "polar_uncharged",
    "N": "polar_uncharged",    "Q": "polar_uncharged",
    "C": "special_cys",
    "K": "positive",           "R": "positive",           "H": "positive",
    "D": "negative",           "E": "negative",
}


def chemical_class(aa: str) -> str:
    return CHEMICAL_CLASS.get(aa, "unknown")


# ── FASTA helpers ─────────────────────────────────────────────────────────────

def parse_fasta(path: Path):
    header, seq_lines = None, []
    with open(path) as fh:
        for line in fh:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if header is not None:
                    yield header, "".join(seq_lines)
                header = line
                seq_lines = []
            else:
                seq_lines.append(line.strip())
    if header is not None:
        yield header, "".join(seq_lines)


def index_fasta(path: Path) -> dict:
    idx = {}
    for header, seq in parse_fasta(path):
        parts = header.lstrip(">").split("|")
        acc = parts[0]
        mut = parts[2] if len(parts) > 2 else "WT"
        idx[f"{acc}_{mut}"] = (header, seq)
    return idx


# ── Position-based clustering ─────────────────────────────────────────────────

def cluster_by_position(df: pd.DataFrame, top_n: int, chemical_diversity: bool
                        ) -> pd.DataFrame:
    """
    For each (accession, position) group, select up to top_n representatives.

    Selection order within each group:
      1. Sort by BLOSUM62 score descending (higher = more plausible substitution)
      2. If chemical_diversity=True, skip candidates whose mut_aa chemical class
         is already represented in the selected set for this position
      3. Stop once top_n representatives are selected

    Returns a filtered DataFrame of selected representatives.
    """
    selected_rows = []

    # Group by enzyme × position
    groups = df.groupby(["accession", "position"], sort=False)

    for (acc, pos), group in groups:
        # Sort by BLOSUM62 score descending; break ties by alphabetical mut_aa
        # (deterministic ordering for reproducibility)
        group_sorted = group.sort_values(
            ["blosum62_score", "mut_aa"], ascending=[False, True]
        )

        if not chemical_diversity or top_n == 1:
            # Simple top-N: just take the first top_n rows
            selected_rows.append(group_sorted.head(top_n))
        else:
            # Chemical-class-diverse top-N
            seen_classes = set()
            picked = []
            for _, row in group_sorted.iterrows():
                cls = chemical_class(row["mut_aa"])
                if cls not in seen_classes:
                    seen_classes.add(cls)
                    picked.append(row)
                    if len(picked) == top_n:
                        break
            if picked:
                selected_rows.append(pd.DataFrame(picked))

    return pd.concat(selected_rows, ignore_index=True)


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    for p in [Path(FILTERED_TSV), Path(FILTERED_FASTA)]:
        if not p.exists():
            print(f"[error] File not found: {p}")
            print("        Run filter_variants.py first.")
            return

    OUT_DIR.mkdir(parents=True, exist_ok=True)

    # 1. Load filtered candidates
    print(f"Loading {FILTERED_TSV} ...")
    df = pd.read_csv(FILTERED_TSV, sep="\t")
    print(f"  {len(df)} filtered candidates across {df['accession'].nunique()} enzymes")
    print(f"  Strategy: top-{TOP_N} per (enzyme, position)"
          + (" with chemical class diversity" if CHEMICAL_DIVERSITY and TOP_N > 1 else ""))

    # 2. Add chemical class columns for reporting
    df["wt_class"]  = df["wt_aa"].apply(chemical_class)
    df["mut_class"] = df["mut_aa"].apply(chemical_class)

    # 3. Cluster
    print("\nClustering by position ...")
    df_reps = cluster_by_position(df, TOP_N, CHEMICAL_DIVERSITY)
    print(f"  {len(df)} → {len(df_reps)} representatives")

    # 4. Report
    sep = "=" * 62
    print(f"\nCLUSTERING REPORT")
    print(sep)
    print(f"  {'Input (filtered variants):':<38} {len(df):>5}")
    print(f"  {'Output (representatives):':<38} {len(df_reps):>5}")
    print(f"  {'Reduction:':<38} {100*(1-len(df_reps)/len(df)):.1f}%")
    print(f"  {'TOP_N:':<38} {TOP_N}")
    print(f"  {'Chemical diversity filter:':<38} {CHEMICAL_DIVERSITY}")

    print(f"\n  Per-enzyme representative counts:")
    for acc, grp in df_reps.groupby("accession"):
        enzyme     = grp["enzyme"].iloc[0]
        n_pos      = grp["position"].nunique()
        n_reps     = len(grp)
        total_pos  = df[df["accession"] == acc]["position"].nunique()
        print(f"    {acc} ({enzyme:<6}): {n_reps:>4} reps across {n_pos}/{total_pos} positions")

    print(f"\n  Chemical class distribution of substitutions (representatives):")
    class_counts = df_reps["mut_class"].value_counts()
    for cls, count in class_counts.items():
        bar = "█" * (count // 10)
        print(f"    {cls:<22}: {count:>4}  {bar}")

    print(f"\n  BLOSUM62 score distribution (representatives):")
    for score in sorted(df_reps["blosum62_score"].unique()):
        n = int((df_reps["blosum62_score"] == score).sum())
        bar = "█" * (n // 5)
        print(f"    {score:>3}: {n:>4}  {bar}")

    print(sep)

    # 5. Write TSV
    tsv_out = OUT_DIR / "clustered_representatives.tsv"
    df_reps.to_csv(tsv_out, sep="\t", index=False)
    print(f"\nTSV:    {tsv_out}")

    # 6. Write report
    report = [
        f"Input filtered variants: {len(df)}",
        f"Output representatives:  {len(df_reps)}",
        f"TOP_N={TOP_N}, chemical_diversity={CHEMICAL_DIVERSITY}",
    ]
    for acc, grp in df_reps.groupby("accession"):
        report.append(f"  {acc}: {len(grp)} representatives")
    (OUT_DIR / "clustering_report.txt").write_text("\n".join(report) + "\n")
    print(f"Report: {OUT_DIR / 'clustering_report.txt'}")

    # 7. Write FASTA
    print(f"Indexing filtered FASTA ...")
    fasta_idx = index_fasta(Path(FILTERED_FASTA))
    fasta_out = OUT_DIR / "clustered_representatives.fasta"
    found, missing = 0, 0
    with open(fasta_out, "w") as fa:
        for _, row in df_reps.iterrows():
            key = f"{row['accession']}_{row['wt_aa']}{row['position']}{row['mut_aa']}"
            if key in fasta_idx:
                header, seq = fasta_idx[key]
                fa.write(f"{header}\n{seq}\n")
                found += 1
            else:
                missing += 1
    print(f"FASTA:  {fasta_out}  ({found} written, {missing} missing)")

    print(f"\nDone.")
    n_af3_jobs = found * 3  # each rep × 3 target enzymes
    print(f"\nNEXT STEP — AlphaFold3 complex prediction:")
    print(f"  {found} representatives × 3 target enzymes = {n_af3_jobs} complex prediction jobs")
    print(f"  Each job: [candidate propeptide] + [target enzyme catalytic domain]")
    print(f"  Rank by ipTM (interface confidence) then pTM (overall fold confidence)")
    print(f"  ipTM > 0.7 = strong predicted interface")
    print(f"  ipTM > 0.8 = high confidence hit — prioritise for wet lab validation")


if __name__ == "__main__":
    main()

Loading output/filtered/filtered_candidates.tsv ...
  5080 filtered candidates across 3 enzymes
  Strategy: top-2 per (enzyme, position) with chemical class diversity

Clustering by position ...
  5080 → 1158 representatives

CLUSTERING REPORT
  Input (filtered variants):              5080
  Output (representatives):               1158
  Reduction:                             77.2%
  TOP_N:                                 2
  Chemical diversity filter:             True

  Per-enzyme representative counts:
    P28784 (RgpA  ):  384 reps across 192/192 positions
    P95493 (RgpB  ):  388 reps across 194/194 positions
    Q51817 (Kgp   ):  386 reps across 193/193 positions

  Chemical class distribution of substitutions (representatives):
    nonpolar_aliphatic    :  355  ███████████████████████████████████
    polar_uncharged       :  304  ██████████████████████████████
    negative              :  207  ████████████████████
    positive              :  158  ███████████████
    aromatic  

## Stage 3bii. MMseqs2-GPU Clustering

In [ ]:
import os
os.environ["PATH"] = "/home/ubuntu/tiffany/mmseqs/bin:" + os.environ["PATH"]

In [23]:
#!/usr/bin/env python3
"""
cluster_candidates_mmseqs2.py
==============================
Stage 3 (database track) of the gingipain inhibitor discovery pipeline.

Input:  output/candidates_db/candidates_filtered.fasta
        output/candidates_db/candidates_filtered_meta.tsv
        (both from filter_candidates_gpu.py)
Output: output/candidates_db/clustered/representatives.fasta
        output/candidates_db/clustered/cluster_summary.tsv

WHY THIS STEP EXISTS
--------------------
filter_candidates_gpu.py only removes EXACT duplicates against the variant
pool. It does nothing about near-identical natural sequences within the
database-collected set itself -- e.g. the same inhibitor domain annotated
independently in five mammalian orthologs will pass the filter as five
"different" candidates, each burning a separate AlphaFold3 job for
essentially the same fold. This step collapses those into one representative
per cluster.

This is the step you already correctly identified as belonging to the
database track and NOT the variant track: point-substitution variants sit
at ~99.5% pairwise identity to their seed by construction, so clustering
would collapse nearly everything. Database candidates have real sequence
diversity, so a 90%-identity cluster threshold is doing real, meaningful
dedup work here instead.

REPRESENTATIVE SELECTION: WHY NOT JUST USE MMSEQS2's DEFAULT PICK
--------------------------------------------------------------------
MMseqs2 picks a default representative per cluster (longest sequence, or
first-encountered, depending on mode) -- it has no concept of your
priority_score (source_tag weight + sweet-spot length + neutral-charge
bonus from the previous filtering stage). This script re-reads
candidates_filtered_meta.tsv after clustering and re-picks the
HIGHEST-priority_score member of each cluster as the actual representative
fed forward to AlphaFold3, overriding MMseqs2's default choice where they
differ. Every cluster's full membership and the runner-up scores are still
recorded in cluster_summary.tsv so nothing is silently discarded.

GPU NOTE (CORRECTED)
--------------------
MMseqs2's `cluster` module does not support GPU acceleration -- per MMseqs2's
own documentation, GPU support applies only to `search`/`easy-search` (via
makepaddedseqdb + --gpu 1). An earlier version of this script incorrectly
passed --gpu 1 to `cluster`, which will always fail regardless of build,
driver, or CUDA setup. This version runs clustering on CPU only, which is
fast enough at a few-thousand-sequence scale (seconds to low minutes) that
GPU wouldn't have meaningfully helped even if it were supported here.
"""

import shutil
import subprocess
import sys
from pathlib import Path
from datetime import datetime

import pandas as pd

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
CONFIG = {
    "input_fasta": Path("output/candidates_db/candidates_filtered.fasta"),
    "meta_tsv": Path("output/candidates_db/candidates_filtered_meta.tsv"),
    "work_dir": Path("output/candidates_db/clustered"),
    "tmp_dir": Path("output/candidates_db/clustered/mmseqs_tmp"),

    # Clustering thresholds
    "min_seq_id": 0.9,   # 90% identity -- collapses near-identical orthologs/isoforms
    "coverage": 0.8,      # 80% alignment coverage required
    "cov_mode": 0,        # 0 = coverage of both query and target

    "threads": 8,
}


# ---------------------------------------------------------------------------
# MMSEQS2 WRAPPER
# ---------------------------------------------------------------------------
def flush_log(cfg: dict, log_lines: list):
    """Write accumulated log lines to disk immediately. Called before every
    sys.exit(1) so a failed run leaves a fresh, accurate log instead of
    silently leaving the previous successful run's log file in place."""
    cfg["work_dir"].mkdir(parents=True, exist_ok=True)
    log_out = cfg["work_dir"] / "clustering_log.txt"
    with open(log_out, "w") as f:
        f.write("\n".join(log_lines) + "\n")
    print(f"  (log written to {log_out} before exiting)")


def check_mmseqs_available():
    if shutil.which("mmseqs") is None:
        print("ERROR: 'mmseqs' executable not found on PATH.")
        print("Install with: conda install -c bioconda mmseqs2")
        print("or:           pip install mmseqs2  (wheel availability varies by platform)")
        sys.exit(1)


def run_cmd(cmd: list, log_lines: list, description: str):
    log_lines.append(f"$ {' '.join(str(c) for c in cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    log_lines.append(result.stdout.strip())
    if result.returncode != 0:
        log_lines.append(f"[STDERR] {result.stderr.strip()}")
    return result


def run_mmseqs_pipeline(cfg: dict, log_lines: list):
    work_dir = cfg["work_dir"]
    tmp_dir = cfg["tmp_dir"]
    work_dir.mkdir(parents=True, exist_ok=True)
    tmp_dir.mkdir(parents=True, exist_ok=True)

    db_path = work_dir / "candidatesDB"
    clu_path = work_dir / "candidatesDB_clu"
    tsv_path = work_dir / "cluster_raw.tsv"

    # 1. Create sequence database
    print("Creating MMseqs2 database...")
    run_cmd(
        ["mmseqs", "createdb", str(cfg["input_fasta"]), str(db_path)],
        log_lines, "createdb",
    )

    # mmseqs cluster (unlike createdb) refuses to overwrite existing output
    # DB files from a previous run -- clean up any stale candidatesDB_clu*
    # files first so re-runs don't collide.
    stale_files = list(work_dir.glob(f"{clu_path.name}*"))
    if stale_files:
        print(f"Removing {len(stale_files)} stale output file(s) from a previous run...")
        for p in stale_files:
            p.unlink()
        log_lines.append(f"Removed stale cluster output files: {[str(p) for p in stale_files]}")

    # 2. Cluster.
    # NOTE: MMseqs2's own documentation is explicit that the `cluster` module
    # does NOT support GPU acceleration -- GPU support applies only to
    # `search`/`easy-search` (via makepaddedseqdb + --gpu 1). Passing --gpu 1
    # to `cluster` will fail regardless of your build, driver, or CUDA setup;
    # this isn't a "your GPU isn't working" problem. At a few thousand short
    # peptides, CPU clustering is fast enough (seconds to low minutes) that
    # GPU wouldn't meaningfully help even if it were supported here -- GPU
    # MMseqs2 pays off at tens-of-millions-of-sequences scale (e.g. UniRef90),
    # not at this dataset size.
    cluster_cmd = [
        "mmseqs", "cluster", str(db_path), str(clu_path), str(tmp_dir),
        "--min-seq-id", str(cfg["min_seq_id"]),
        "-c", str(cfg["coverage"]),
        "--cov-mode", str(cfg["cov_mode"]),
        "--threads", str(cfg["threads"]),
    ]
    print("Clustering (CPU -- MMseqs2's cluster module has no GPU path)...")
    result = run_cmd(cluster_cmd, log_lines, "cluster (CPU)")
    if result.returncode != 0:
        print("ERROR: mmseqs cluster failed. Check log for stderr details.")
        flush_log(cfg, log_lines)
        sys.exit(1)

    # 3. Export cluster assignments as TSV (representative_id \t member_id per row)
    if tsv_path.exists():
        tsv_path.unlink()
    print("Exporting cluster assignments...")
    run_cmd(
        ["mmseqs", "createtsv", str(db_path), str(db_path), str(clu_path), str(tsv_path)],
        log_lines, "createtsv",
    )

    return tsv_path


# ---------------------------------------------------------------------------
# FASTA I/O
# ---------------------------------------------------------------------------
def read_fasta_by_id(path: Path):
    """Read FASTA -> dict of accession -> sequence, keyed on the accession
    portion before the first '|' (matches the header format written by
    filter_candidates_gpu.py: >accession|source_tag|score=...)."""
    seqs = {}
    header_id = None
    chunks = []
    with open(path) as f:
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            if line.startswith(">"):
                if header_id is not None:
                    seqs[header_id] = "".join(chunks)
                header_id = line[1:].split("|")[0]
                chunks = []
            else:
                chunks.append(line.strip())
    if header_id is not None:
        seqs[header_id] = "".join(chunks)
    return seqs


# ---------------------------------------------------------------------------
# REPRESENTATIVE RESELECTION BY PRIORITY SCORE
# ---------------------------------------------------------------------------
def reselect_representatives(cluster_tsv: Path, meta_tsv: Path, log_lines: list):
    clusters = pd.read_csv(
        cluster_tsv, sep="\t", header=None,
        names=["mmseqs_representative_raw", "member_raw"],
    )

    # IMPORTANT: MMseqs2 assigns sequence IDs by splitting each FASTA header
    # at the first WHITESPACE character, not at '|'. Our headers
    # (">accession|source_tag|score=...") contain no whitespace, so MMseqs2
    # stored the ENTIRE header line as the ID. Extract just the accession
    # (text before the first '|') so this can actually join against
    # meta_tsv's accession-only column.
    clusters["mmseqs_representative"] = clusters["mmseqs_representative_raw"].str.split("|").str[0]
    clusters["member"] = clusters["member_raw"].str.split("|").str[0]

    meta = pd.read_csv(meta_tsv, sep="\t")

    if "priority_score" not in meta.columns:
        log_lines.append("WARNING: priority_score column not found in meta_tsv -- "
                          "falling back to MMseqs2's default representative choice.")
        meta["priority_score"] = 0.0

    merged = clusters.merge(
        meta, left_on="member", right_on="accession", how="left"
    )
    merged["priority_score"] = merged["priority_score"].fillna(0.0)

    n_missing_meta = merged["accession"].isna().sum()
    if n_missing_meta:
        log_lines.append(f"WARNING: {n_missing_meta} cluster members had no matching "
                          f"row in meta_tsv (priority_score defaulted to 0.0).")

    # Cluster size per mmseqs_representative group
    cluster_sizes = merged.groupby("mmseqs_representative")["member"].transform("count")
    merged["cluster_size"] = cluster_sizes

    # Pick highest-priority_score member per cluster as the FINAL representative.
    # Ties broken by longer sequence (via 'length' column if present), then by
    # accession string for determinism.
    sort_cols = ["mmseqs_representative", "priority_score"]
    ascending = [True, False]
    if "length" in merged.columns:
        sort_cols.append("length")
        ascending.append(False)
    sort_cols.append("member")
    ascending.append(True)

    merged_sorted = merged.sort_values(sort_cols, ascending=ascending)
    final_reps = merged_sorted.groupby("mmseqs_representative", as_index=False).first()

    n_overridden = (final_reps["member"] != final_reps["mmseqs_representative"]).sum()
    log_lines.append(f"Clusters: {len(final_reps)}")
    log_lines.append(f"Representative overridden from MMseqs2's default pick "
                      f"(higher-priority member chosen instead): {n_overridden} / {len(final_reps)}")

    final_reps = final_reps.rename(columns={"member": "final_representative"})
    return final_reps, merged


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------
def main():
    cfg = CONFIG
    log_lines = [f"cluster_candidates_mmseqs2.py run: {datetime.now().isoformat()}"]

    check_mmseqs_available()

    if not cfg["input_fasta"].exists():
        print(f"ERROR: input FASTA not found at {cfg['input_fasta']}. "
              f"Run filter_candidates_gpu.py first.")
        sys.exit(1)
    if not cfg["meta_tsv"].exists():
        print(f"ERROR: metadata TSV not found at {cfg['meta_tsv']}. "
              f"Run filter_candidates_gpu.py first.")
        sys.exit(1)

    cluster_tsv = run_mmseqs_pipeline(cfg, log_lines)

    print("Re-selecting cluster representatives by priority_score...")
    final_reps, full_membership = reselect_representatives(cluster_tsv, cfg["meta_tsv"], log_lines)

    # Write final representative FASTA
    all_seqs = read_fasta_by_id(cfg["input_fasta"])
    rep_fasta_path = cfg["work_dir"] / "representatives.fasta"
    n_missing_seq = 0
    with open(rep_fasta_path, "w") as f:
        for _, row in final_reps.iterrows():
            acc = row["final_representative"]
            seq = all_seqs.get(acc)
            if seq is None:
                n_missing_seq += 1
                continue
            f.write(f">{acc}|cluster_size={int(row['cluster_size'])}|"
                    f"score={row['priority_score']:.2f}\n")
            f.write(f"{seq}\n")
    if n_missing_seq:
        log_lines.append(f"WARNING: {n_missing_seq} representatives had no matching "
                          f"sequence in input FASTA (skipped in output).")

    # Write full cluster summary (every member, not just representatives)
    summary_path = cfg["work_dir"] / "cluster_summary.tsv"
    rep_lookup = final_reps.set_index("mmseqs_representative")["final_representative"]
    full_membership["final_representative"] = full_membership["mmseqs_representative"].map(rep_lookup)
    full_membership["is_final_representative"] = (
        full_membership["member"] == full_membership["final_representative"]
    )
    out_cols = ["mmseqs_representative", "final_representative", "member",
                "is_final_representative", "cluster_size", "priority_score"]
    out_cols = [c for c in out_cols if c in full_membership.columns]
    full_membership[out_cols].sort_values(
        ["mmseqs_representative", "priority_score"], ascending=[True, False]
    ).to_csv(summary_path, sep="\t", index=False)

    log_out = cfg["work_dir"] / "clustering_log.txt"
    with open(log_out, "w") as f:
        f.write("\n".join(log_lines) + "\n")

    n_input = len(all_seqs)
    n_output = len(final_reps) - n_missing_seq
    print(f"\nDone.")
    print(f"  Input candidates: {n_input}")
    print(f"  Clusters / final representatives: {n_output}")
    print(f"  Reduction: {n_input} -> {n_output} "
          f"({100 * (1 - n_output / n_input):.1f}% collapsed)" if n_input else "")
    print(f"  Representatives FASTA: {rep_fasta_path}")
    print(f"  Cluster summary:       {summary_path}")
    print(f"  Log:                   {log_out}")
    print(f"\nNEXT STEPS:")
    print(f"  1. Spot-check cluster_summary.tsv for any cluster where the highest-")
    print(f"     priority member was NOT MMseqs2's default pick -- confirms the")
    print(f"     override logic is doing something meaningful, not a no-op.")
    print(f"  2. Feed representatives.fasta into AF3 job generation, tagged")
    print(f"     source='database' in the job manifest, alongside variant-track")
    print(f"     candidates tagged source='variant'.")
    print(f"  3. Once both tracks have AF3 results, parse_af3_results.py can compare")
    print(f"     ipTM/pTM distributions by source to see which strategy is finding")
    print(f"     better binders.")


if __name__ == "__main__":
    main()

Creating MMseqs2 database...
Removing 10 stale output file(s) from a previous run...
Clustering (CPU -- MMseqs2's cluster module has no GPU path)...
Exporting cluster assignments...
Re-selecting cluster representatives by priority_score...

Done.
  Input candidates: 3173
  Clusters / final representatives: 2112
  Reduction: 3173 -> 2112 (33.4% collapsed)
  Representatives FASTA: output/candidates_db/clustered/representatives.fasta
  Cluster summary:       output/candidates_db/clustered/cluster_summary.tsv
  Log:                   output/candidates_db/clustered/clustering_log.txt

NEXT STEPS:
  1. Spot-check cluster_summary.tsv for any cluster where the highest-
     priority member was NOT MMseqs2's default pick -- confirms the
     override logic is doing something meaningful, not a no-op.
  2. Feed representatives.fasta into AF3 job generation, tagged
     source='database' in the job manifest, alongside variant-track
     candidates tagged source='variant'.
  3. Once both tracks hav

## Stage 3c. Extracting Catalytic Domains

In [6]:
#!/usr/bin/env python3
"""
extract_catalytic_domains.py
============================
Extracts the catalytic domain (Chain) sequences of RgpA, RgpB, and Kgp
from the locally-downloaded UniProt FASTA files, using verified chain
boundary coordinates from UniProt's PTM/Processing tab.

These catalytic domain sequences are the TARGET structures for AlphaFold3
complex prediction — the propeptide candidate gets paired against each of these.

Why the catalytic domain and not the full precursor?
  The full precursor includes the signal peptide (cleaved during secretion)
  and the propeptide (cleaved during activation). The mature enzyme that
  exists in the periodontal pocket — and that an exogenous inhibitor would
  encounter — is just the catalytic domain (± IgSF adhesin domain for RgpA).
  Feeding AlphaFold3 the full precursor would confuse the prediction with
  regions that don't exist in the active enzyme.

Chain boundaries (from UniProt PTM/Processing tab, "Chain" row):
  P28784 (RgpA): Chain = Gingipain R1,       residues 228–991
  P95493 (RgpB): Chain = Gingipain R2,       residues 230–736
  Q51817 (Kgp):  Chain = Lys-gingipain W83,  residues 229–1732
                  ↑ verify these at uniprot.org before AlphaFold3 submission

Inputs:
  The same UniProt FASTA files used in extract_propeptides.py

Outputs:
  output/targets/catalytic_domains.fasta   — one sequence per canonical enzyme
  output/targets/catalytic_domains.tsv     — accession, enzyme, start, end, length
"""

from pathlib import Path

# ============================================================
# CONFIG — same FASTA files as extract_propeptides.py
# ============================================================
FASTA_FILES = [
    "../datasets/UniProt/uniprotkb_gingipain_RgpA_Porphyromonas_2026_07_02.fasta",
    "../datasets/UniProt/uniprotkb_gingipain_RgpB_Porphyromonas_2026_07_02.fasta",
    "../datasets/UniProt/uniprotkb_gingipain_Kgp_Porphyromonas_g_2026_07_02.fasta",
]
OUT_DIR = Path("output/targets")

# Chain boundaries (1-based, inclusive) from UniProt PTM/Processing tab.
# VERIFY each at uniprot.org/<accession> → PTM/Processing → Chain row
# before using for AlphaFold3 — coordinates differ per strain.
#
# P28784 (RgpA): signal 1-24, propeptide 25-227, chain 228-991  ← CONFIRMED
# P95493 (RgpB): signal 1-24, propeptide 25-229, chain 230-736  ← INFERRED, verify
# Q51817 (Kgp):  signal 1-24, propeptide 25-228, chain 229-1732 ← INFERRED, verify
#
# Format: accession -> (chain_start, chain_end, enzyme_label, chain_name)
CHAIN_COORDS = {
    "P28784": (228,  991,  "RgpA", "Gingipain_R1"),
    "P95493": (230,  736,  "RgpB", "Gingipain_R2"),
    "Q51817": (229, 1732,  "Kgp",  "Lys-gingipain_W83"),
}

# Only extract these canonical accessions (PE=1, reviewed)
CANONICAL = set(CHAIN_COORDS.keys())
# ============================================================


def parse_fasta(path: Path):
    """Yield (accession, header, sequence) from a UniProt FASTA file."""
    header, seq_lines = None, []
    with open(path) as fh:
        for line in fh:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if header is not None:
                    yield _acc(header), header, "".join(seq_lines)
                header = line
                seq_lines = []
            else:
                seq_lines.append(line.strip())
    if header is not None:
        yield _acc(header), header, "".join(seq_lines)


def _acc(header: str) -> str:
    parts = header.lstrip(">").split("|")
    return parts[1] if len(parts) >= 2 else parts[0].split()[0]


def main():
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    seen = set()
    results = []

    for fasta_path in FASTA_FILES:
        path = Path(fasta_path)
        if not path.exists():
            print(f"[skip] file not found: {fasta_path}")
            continue

        print(f"Reading {path.name} ...")
        for acc, header, seq in parse_fasta(path):
            if acc not in CANONICAL or acc in seen:
                continue
            seen.add(acc)

            start, end, enzyme, chain_name = CHAIN_COORDS[acc]

            # Validate coordinates against actual sequence length
            if end > len(seq):
                print(f"  [warn] {acc}: sequence length {len(seq)} < chain end {end}")
                print(f"         Check coordinates — using sequence length as end")
                end = len(seq)

            if start > len(seq):
                print(f"  [error] {acc}: chain start {start} > sequence length {len(seq)}")
                print(f"          Skipping — verify coordinates at uniprot.org/{acc}")
                continue

            # Slice catalytic domain (1-based inclusive → 0-based Python slice)
            domain_seq = seq[start - 1:end]
            domain_len = len(domain_seq)

            print(f"  {acc} ({enzyme}): extracted {domain_len} aa catalytic domain "
                  f"[{start}-{end}] from {len(seq)} aa precursor")

            results.append({
                "accession":   acc,
                "enzyme":      enzyme,
                "chain_name":  chain_name,
                "chain_start": start,
                "chain_end":   end,
                "domain_len":  domain_len,
                "sequence":    domain_seq,
            })

    if not results:
        print("[error] No sequences extracted. Check file paths and accessions.")
        return

    # Write FASTA
    fasta_out = OUT_DIR / "catalytic_domains.fasta"
    with open(fasta_out, "w") as fa:
        for r in results:
            header = (f">{r['accession']}|catalytic_domain|{r['enzyme']}|"
                      f"{r['chain_name']}|{r['chain_start']}-{r['chain_end']}")
            fa.write(f"{header}\n{r['sequence']}\n")
    print(f"\nFASTA: {fasta_out}  ({len(results)} sequences)")

    # Write TSV (without full sequence column for readability)
    tsv_out = OUT_DIR / "catalytic_domains.tsv"
    with open(tsv_out, "w") as tsv:
        tsv.write("accession\tenzyme\tchain_name\tchain_start\tchain_end\tdomain_len\n")
        for r in results:
            tsv.write(
                f"{r['accession']}\t{r['enzyme']}\t{r['chain_name']}\t"
                f"{r['chain_start']}\t{r['chain_end']}\t{r['domain_len']}\n"
            )
    print(f"TSV:   {tsv_out}")

    print(f"\nSummary:")
    for r in results:
        print(f"  {r['accession']} ({r['enzyme']}): {r['domain_len']} aa  "
              f"[{r['chain_name']}, residues {r['chain_start']}–{r['chain_end']}]")

    print(f"\n{'='*58}")
    print(f"BEFORE AlphaFold3 — verify chain coordinates:")
    for r in results:
        print(f"  uniprot.org/{r['accession']} → PTM/Processing → Chain row")
    print(f"{'='*58}")
    print(f"\nNEXT STEP: generate AlphaFold3 input JSONs")
    print(f"  python generate_af3_jobs.py")
    print(f"  Pairs each of 1158 candidates × {len(results)} targets "
          f"= {1158 * len(results)} prediction jobs")


if __name__ == "__main__":
    main()

Reading uniprotkb_gingipain_RgpA_Porphyromonas_2026_07_02.fasta ...
  P95493 (RgpB): extracted 507 aa catalytic domain [230-736] from 736 aa precursor
  P28784 (RgpA): extracted 764 aa catalytic domain [228-991] from 991 aa precursor
Reading uniprotkb_gingipain_RgpB_Porphyromonas_2026_07_02.fasta ...
Reading uniprotkb_gingipain_Kgp_Porphyromonas_g_2026_07_02.fasta ...
  Q51817 (Kgp): extracted 1504 aa catalytic domain [229-1732] from 1732 aa precursor

FASTA: output/targets/catalytic_domains.fasta  (3 sequences)
TSV:   output/targets/catalytic_domains.tsv

Summary:
  P95493 (RgpB): 507 aa  [Gingipain_R2, residues 230–736]
  P28784 (RgpA): 764 aa  [Gingipain_R1, residues 228–991]
  Q51817 (Kgp): 1504 aa  [Lys-gingipain_W83, residues 229–1732]

BEFORE AlphaFold3 — verify chain coordinates:
  uniprot.org/P95493 → PTM/Processing → Chain row
  uniprot.org/P28784 → PTM/Processing → Chain row
  uniprot.org/Q51817 → PTM/Processing → Chain row

NEXT STEP: generate AlphaFold3 input JSONs
  pytho

# Stage 4. Complex Prediction (and or docking)

## Stage 4a. Alphafold

In [ ]:
#!/usr/bin/env python3
"""
generate_af3_jobs.py
====================
Generates AlphaFold3 JSON input files for all candidate propeptide ×
target catalytic domain pairs, then launches them in batches.

AlphaFold3 complex prediction input format:
  Each job is a JSON file describing a multimer prediction:
    - sequence A: candidate propeptide variant (~204 aa)
    - sequence B: target catalytic domain (507–1504 aa)
  AlphaFold3 predicts the complex structure and outputs:
    - ipTM: interface predicted TM-score (confidence in the interface)
    - pTM:  overall predicted TM-score  (confidence in the full fold)

  ipTM > 0.7 → strong predicted interface
  ipTM > 0.8 → high-confidence hit, prioritise for wet lab

Job count:
  1158 candidates × 3 targets = 3474 jobs total
  At ~10 min/job on A100: ~580 GPU-hours
  On 4× A100: ~145 hours wall time if run serially
  Recommended: run in parallel batches (see BATCH_SIZE and launch logic below)

Outputs:
  output/af3_jobs/inputs/  — one JSON per job
  output/af3_jobs/run_af3.sh — shell script to launch all jobs
  output/af3_jobs/job_manifest.tsv — maps job_id to candidate + target
"""

import json
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================
# CHANGE THIS
CANDIDATES_FASTA = "output/topn_clustered/clustered_representatives.fasta"
TARGETS_FASTA    = "output/targets/catalytic_domains.fasta"
OUT_DIR          = Path("output/af3_jobs")

# Path to your AlphaFold3 installation on the server
AF3_SCRIPT = "/home/ubuntu/alphafold3/run_alphafold.py"

# AlphaFold3 model weights and database directories — update to your paths
AF3_MODEL_DIR = "/data/af3_models"
AF3_DB_DIR    = "/data/af3_databases"

# Number of seeds per job (AF3 runs multiple seeds and picks best).
# 5 is the AF3 default; reduce to 1 for faster screening, increase to 10
# for final validation of top hits.
NUM_SEEDS = 5

# GPU to use (single GPU per job). For multi-GPU parallelism,
# the launch script assigns jobs round-robin across available GPUs.
NUM_GPUS = 4   # set to however many GPUs your server has
# ============================================================


def parse_fasta(path: Path):
    """Yield (header, sequence) from a FASTA file."""
    header, seq_lines = None, []
    with open(path) as fh:
        for line in fh:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if header is not None:
                    yield header, "".join(seq_lines)
                header = line
                seq_lines = []
            else:
                seq_lines.append(line.strip())
    if header is not None:
        yield header, "".join(seq_lines)


def parse_candidate_header(header: str) -> dict:
    """
    Parse '>P28784|variant|Y25A|RgpA|25-227' into metadata dict.
    """
    parts = header.lstrip(">").split("|")
    return {
        "accession": parts[0] if len(parts) > 0 else "UNK",
        "mut_label": parts[2] if len(parts) > 2 else "UNK",  # e.g. Y25A
        "enzyme":    parts[3] if len(parts) > 3 else "UNK",
        "coord":     parts[4] if len(parts) > 4 else "UNK",
    }


def parse_target_header(header: str) -> dict:
    """
    Parse '>P28784|catalytic_domain|RgpA|Gingipain_R1|228-991' into metadata.
    """
    parts = header.lstrip(">").split("|")
    return {
        "accession":  parts[0] if len(parts) > 0 else "UNK",
        "enzyme":     parts[2] if len(parts) > 2 else "UNK",
        "chain_name": parts[3] if len(parts) > 3 else "UNK",
        "coord":      parts[4] if len(parts) > 4 else "UNK",
    }


def make_af3_json(job_id: str,
                  candidate_seq: str,
                  target_seq: str,
                  candidate_meta: dict,
                  target_meta: dict,
                  num_seeds: int) -> dict:
    """
    Build an AlphaFold3 JSON input dict for one candidate × target pair.

    AlphaFold3 multimer input format:
      "name"     : job identifier (used as output directory name)
      "sequences": list of chain dicts, each with "protein" key
      "modelSeeds": list of integer seeds
    """
    return {
        "name": job_id,
        "modelSeeds": list(range(num_seeds)),
        "sequences": [
            {
                "protein": {
                    "id": "A",
                    "sequence": candidate_seq,
                    "description": (
                        f"candidate|{candidate_meta['accession']}|"
                        f"{candidate_meta['mut_label']}|{candidate_meta['enzyme']}"
                    ),
                }
            },
            {
                "protein": {
                    "id": "B",
                    "sequence": target_seq,
                    "description": (
                        f"target|{target_meta['accession']}|"
                        f"{target_meta['enzyme']}|{target_meta['chain_name']}"
                    ),
                }
            },
        ],
    }


def main():
    for p in [Path(CANDIDATES_FASTA), Path(TARGETS_FASTA)]:
        if not p.exists():
            print(f"[error] File not found: {p}")
            return

    inputs_dir = OUT_DIR / "inputs"
    inputs_dir.mkdir(parents=True, exist_ok=True)

    # 1. Load candidates and targets
    print(f"Loading candidates from {CANDIDATES_FASTA} ...")
    candidates = [
        (parse_candidate_header(h), seq)
        for h, seq in parse_fasta(Path(CANDIDATES_FASTA))
    ]
    print(f"  {len(candidates)} candidate sequences")

    print(f"Loading targets from {TARGETS_FASTA} ...")
    targets = [
        (parse_target_header(h), seq)
        for h, seq in parse_fasta(Path(TARGETS_FASTA))
    ]
    print(f"  {len(targets)} target sequences")

    total_jobs = len(candidates) * len(targets)
    print(f"\nGenerating {len(candidates)} × {len(targets)} = {total_jobs} JSON input files ...")

    # 2. Generate one JSON per (candidate, target) pair
    manifest_rows = []

    for c_meta, c_seq in candidates:
        for t_meta, t_seq in targets:
            # Job ID encodes candidate accession, mutation, and target enzyme
            # e.g. "P28784_Y25A_vs_RgpB"
            job_id = (
                f"{c_meta['accession']}_{c_meta['mut_label']}"
                f"_vs_{t_meta['enzyme']}"
            )

            job_json = make_af3_json(
                job_id        = job_id,
                candidate_seq = c_seq,
                target_seq    = t_seq,
                candidate_meta = c_meta,
                target_meta    = t_meta,
                num_seeds      = NUM_SEEDS,
            )

            json_path = inputs_dir / f"{job_id}.json"
            with open(json_path, "w") as f:
                json.dump(job_json, f, indent=2)

            manifest_rows.append({
                "job_id":           job_id,
                "candidate_acc":    c_meta["accession"],
                "mut_label":        c_meta["mut_label"],
                "candidate_enzyme": c_meta["enzyme"],
                "target_acc":       t_meta["accession"],
                "target_enzyme":    t_meta["enzyme"],
                "json_path":        str(json_path),
            })

    print(f"  Written {len(manifest_rows)} JSON files to {inputs_dir}")

    # 3. Write manifest TSV
    manifest_path = OUT_DIR / "job_manifest.tsv"
    with open(manifest_path, "w") as tsv:
        tsv.write("\t".join(manifest_rows[0].keys()) + "\n")
        for row in manifest_rows:
            tsv.write("\t".join(str(v) for v in row.values()) + "\n")
    print(f"  Manifest: {manifest_path}")

    # 4. Generate launch shell script
    # Assigns jobs round-robin across NUM_GPUS GPUs so all GPUs stay busy.
    launch_path = OUT_DIR / "run_af3.sh"
    json_paths  = [str(inputs_dir / f"{r['job_id']}.json") for r in manifest_rows]

    with open(launch_path, "w") as sh:
        sh.write("#!/bin/bash\n")
        sh.write("# AlphaFold3 batch launch script\n")
        sh.write(f"# {total_jobs} jobs across {NUM_GPUS} GPUs\n")
        sh.write("# Usage: bash run_af3.sh\n\n")
        sh.write("set -euo pipefail\n\n")
        sh.write(f"AF3_SCRIPT={AF3_SCRIPT}\n")
        sh.write(f"MODEL_DIR={AF3_MODEL_DIR}\n")
        sh.write(f"DB_DIR={AF3_DB_DIR}\n")
        sh.write(f"OUTPUT_BASE={OUT_DIR / 'outputs'}\n\n")
        sh.write("mkdir -p \"$OUTPUT_BASE\"\n\n")
        sh.write("PIDS=()\n\n")

        for i, json_path in enumerate(json_paths):
            gpu_id   = i % NUM_GPUS
            job_name = manifest_rows[i]["job_id"]
            out_dir  = f"$OUTPUT_BASE/{job_name}"
            sh.write(
                f"CUDA_VISIBLE_DEVICES={gpu_id} python \"$AF3_SCRIPT\" \\\n"
                f"    --json_path={json_path} \\\n"
                f"    --output_dir={out_dir} \\\n"
                f"    --model_dir=\"$MODEL_DIR\" \\\n"
                f"    --db_dir=\"$DB_DIR\" &\n"
            )
            # Collect PIDs and wait every NUM_GPUS jobs to avoid overloading
            sh.write(f"PIDS+=($!)\n")
            if (i + 1) % NUM_GPUS == 0:
                sh.write(f"# Wait for this batch of {NUM_GPUS} to finish\n")
                sh.write("for PID in \"${PIDS[@]}\"; do wait \"$PID\"; done\n")
                sh.write("PIDS=()\n\n")

        # Wait for any remaining jobs
        sh.write("\n# Wait for final batch\n")
        sh.write("for PID in \"${PIDS[@]}\"; do wait \"$PID\"; done\n")
        sh.write('\necho "All AlphaFold3 jobs complete."\n')

    launch_path.chmod(0o755)
    print(f"  Launch script: {launch_path}")

    # 5. Print summary
    print(f"\n{'='*58}")
    print(f"SUMMARY")
    print(f"{'='*58}")
    print(f"  Total jobs:        {total_jobs}")
    print(f"  JSON inputs:       {inputs_dir}")
    print(f"  Launch script:     {launch_path}")
    print(f"  Manifest:          {manifest_path}")
    print(f"\nBEFORE RUNNING:")
    print(f"  1. Update AF3_SCRIPT, AF3_MODEL_DIR, AF3_DB_DIR in CONFIG")
    print(f"     to match your server's AlphaFold3 installation paths")
    print(f"  2. Verify NUM_GPUS={NUM_GPUS} matches your server")
    print(f"  3. Consider running a single test job first:")
    print(f"     CUDA_VISIBLE_DEVICES=0 python {AF3_SCRIPT} \\")
    print(f"         --json_path={json_paths[0]} \\")
    print(f"         --output_dir={OUT_DIR}/outputs/test \\")
    print(f"         --model_dir={AF3_MODEL_DIR} \\")
    print(f"         --db_dir={AF3_DB_DIR}")
    print(f"\nAFTER RUNNING:")
    print(f"  python parse_af3_results.py")
    print(f"  Collects ipTM/pTM scores from all output directories,")
    print(f"  ranks candidates, flags hits with ipTM > 0.7")


if __name__ == "__main__":
    main()

## Stage 4b. ColabFold

In [2]:
#!/usr/bin/env python3
"""
generate_colabfold_jobs.py
============================
Interim stand-in for generate_af3_jobs.py while waiting on AlphaFold3
parameter access. Generates ColabFold multimer input FASTAs pairing both
candidate tracks (variant + database) against the 3 gingipain catalytic
domain targets, plus a launch script and job manifest.

MODE: PUBLIC API (not local databases)
-----------------------------------------
This version targets colabfold_batch's DEFAULT behavior, which sends MSA
queries to the public ColabFold MMseqs2 API server (api.colabfold.com) --
it does NOT require setup_databases.sh or a local sequence database. This
is the right choice for an initial test run: no multi-hundred-GB database
download, no colabfold_search step, just colabfold_batch directly.

The public server is a shared resource (documented as capped around a few
thousand MSA queries/day across all users), so this mode is only
appropriate for a SMALL test slice, not the full 9,810-job run -- use
--limit to keep it small until you've verified the output looks right,
then move to local databases (setup_databases.sh + colabfold_search) for
the full-scale run.

WHY COLABFOLD IS A REASONABLE STAND-IN HERE
---------------------------------------------
ColabFold predicts structures with AlphaFold2 (not AF3), using MMseqs2 for
the MSA step instead of HHblits/jackhmmer -- that's what makes it fast
enough to run practically on a single GPU. For candidate x target complex
prediction, colabfold_batch's multimer mode gives you `iptm` and `plddt`
per prediction, which is directionally comparable to AF3's ipTM/pTM for
RANKING candidates against each other -- not numerically identical to AF3
output, so treat this as a pre-screen, not a substitute for the eventual
AF3 pass.

INPUT FORMAT DIFFERENCE FROM AF3
----------------------------------
AF3 takes one JSON per job. ColabFold's `colabfold_batch` takes a FASTA
file where each record is one job; chains within a complex are joined with
':' in the sequence. This script writes one combined FASTA per track
(so colabfold_batch can process many jobs from one input file) plus a
manifest TSV mapping job_id -> (candidate accession, target, source_track)
for the results parser to join back against.

INPUTS
------
  output/propeptides/clustered_representatives.fasta   (variant track, 1158)
  output/candidates_db/clustered/representatives.fasta  (database track, 2112)
  output/targets/catalytic_domains.fasta                (3 targets: RgpA/RgpB/Kgp)

OUTPUTS
-------
  output/colabfold_jobs/colabfold_input_variant.fasta
  output/colabfold_jobs/colabfold_input_database.fasta
  output/colabfold_jobs/job_manifest.tsv
  output/colabfold_jobs/run_colabfold.sh   (round-robin multi-GPU launcher)

USAGE
-----
  python3 generate_colabfold_jobs.py                 # uses CONFIG default (small test)
  python3 generate_colabfold_jobs.py --limit 3        # first 3 candidates per track (quick test)
  python3 generate_colabfold_jobs.py --limit 50       # first 50 candidates per track
  python3 generate_colabfold_jobs.py --limit all      # entire file, both tracks (NOT recommended
                                                       # against the public API -- switch to local
                                                       # databases first, see module docstring above)
"""

import re
import argparse
from pathlib import Path
from datetime import datetime

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
CONFIG = {
    "variant_candidates_fasta": Path("output/topn_clustered/clustered_representatives.fasta"),
    "database_candidates_fasta": Path("output/candidates_db/clustered/representatives.fasta"),
    "targets_fasta": Path("output/targets/catalytic_domains.fasta"),

    "output_dir": Path("output/colabfold_jobs"),

    # Candidates-per-track cap. Defaults to a small test size since this is
    # the PUBLIC API mode (see module docstring) -- override at runtime with
    # --limit rather than editing this every time. 1158 variant x 3 +
    # 2112 database x 3 = 9,810 total jobs if uncapped, which is far more
    # than the public MSA server is meant to absorb in one run. Set to None
    # here (or pass --limit all) only once you've moved to local databases.
    #
    # Note: database track candidates are already sorted by priority_score
    # (from cluster_candidates_mmseqs2.py), so a cap there is a true top-N.
    # The variant track FASTA isn't score-sorted, so "first N" just follows
    # whatever order clustering wrote it in -- sort that file first if you
    # want a specific priority order for the variant track's test slice too.
    "candidates_per_track_limit": 3,

    # Multi-GPU round-robin launch script generation
    "n_gpus": 1,
    "colabfold_batch_extra_args": "--num-models 1 --num-recycle 3",
}


# ---------------------------------------------------------------------------
# FASTA I/O
# ---------------------------------------------------------------------------
def read_fasta(path: Path):
    """FASTA -> list of (header, sequence)."""
    entries = []
    header = None
    chunks = []
    with open(path) as f:
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    entries.append((header, "".join(chunks)))
                header = line[1:]
                chunks = []
            else:
                chunks.append(line.strip())
    if header is not None:
        entries.append((header, "".join(chunks)))
    return entries


def clean_job_id(raw: str) -> str:
    """ColabFold uses the FASTA header as the output filename prefix --
    strip characters that are awkward in filenames (|, =, spaces)."""
    return re.sub(r"[^A-Za-z0-9_\-.]", "_", raw)


# ---------------------------------------------------------------------------
# JOB GENERATION
# ---------------------------------------------------------------------------
def generate_track_jobs(candidates: list, targets: list, track_name: str,
                         limit: int, manifest_rows: list):
    """Pair every candidate with every target, return list of (job_id, complex_seq).
    limit=None means no cap (use every candidate)."""
    if limit is not None:
        candidates = candidates[:limit]

    jobs = []
    for cand_header, cand_seq in candidates:
        cand_acc = cand_header.split("|")[0]
        for tgt_header, tgt_seq in targets:
            tgt_id = tgt_header.split("|")[0].split()[0]
            job_id = clean_job_id(f"{cand_acc}__vs__{tgt_id}")
            complex_seq = f"{cand_seq}:{tgt_seq}"
            jobs.append((job_id, complex_seq))
            manifest_rows.append({
                "job_id": job_id,
                "candidate_accession": cand_acc,
                "target_id": tgt_id,
                "source_track": track_name,
                "candidate_length": len(cand_seq),
                "target_length": len(tgt_seq),
            })
    return jobs


def write_jobs_fasta(jobs: list, path: Path):
    with open(path, "w") as f:
        for job_id, seq in jobs:
            f.write(f">{job_id}\n{seq}\n")


# ---------------------------------------------------------------------------
# LAUNCH SCRIPT
# ---------------------------------------------------------------------------
def write_launch_script(cfg: dict, path: Path, track_files: list):
    """Round-robin GPU launcher. Splits each track's input FASTA into
    per-GPU chunks and backgrounds one colabfold_batch process per GPU,
    mirroring the AF3 multi-GPU launcher pattern."""
    n_gpus = cfg["n_gpus"]
    lines = [
        "#!/bin/bash",
        "# Auto-generated by generate_colabfold_jobs.py",
        "# Splits each track's job FASTA across available GPUs and runs",
        "# colabfold_batch in parallel, one process per GPU.",
        "set -e",
        "",
        f"N_GPUS={n_gpus}",
        f'EXTRA_ARGS="{cfg["colabfold_batch_extra_args"]}"',
        "",
    ]

    for track_name, fasta_path in track_files:
        out_subdir = cfg["output_dir"] / f"results_{track_name}"
        split_dir = cfg["output_dir"] / f"split_{track_name}"
        lines.append(f'echo "=== {track_name} track ==="')
        lines.append(f"mkdir -p {split_dir} {out_subdir}")
        lines.append(
            f"python3 -c \"\n"
            f"import sys\n"
            f"from pathlib import Path\n"
            f"entries = []\n"
            f"header = None; chunks = []\n"
            f"with open('{fasta_path}') as f:\n"
            f"    for line in f:\n"
            f"        line = line.rstrip()\n"
            f"        if not line: continue\n"
            f"        if line.startswith('>'):\n"
            f"            if header is not None: entries.append((header, ''.join(chunks)))\n"
            f"            header = line[1:]; chunks = []\n"
            f"        else:\n"
            f"            chunks.append(line)\n"
            f"    if header is not None: entries.append((header, ''.join(chunks)))\n"
            f"n_gpus = {n_gpus}\n"
            f"buckets = [[] for _ in range(n_gpus)]\n"
            f"for i, e in enumerate(entries): buckets[i % n_gpus].append(e)\n"
            f"for gpu_idx, bucket in enumerate(buckets):\n"
            f"    with open('{split_dir}/gpu' + str(gpu_idx) + '.fasta', 'w') as out:\n"
            f"        for h, s in bucket: out.write('>' + h + '\\\\n' + s + '\\\\n')\n"
            f"\""
        )
        lines.append(f"for gpu_idx in $(seq 0 $((N_GPUS - 1))); do")
        lines.append(
            f'  CUDA_VISIBLE_DEVICES=$gpu_idx colabfold_batch $EXTRA_ARGS '
            f'"{split_dir}/gpu${{gpu_idx}}.fasta" "{out_subdir}" '
            f'> "{cfg["output_dir"]}/log_{track_name}_gpu${{gpu_idx}}.txt" 2>&1 &'
        )
        lines.append("done")
        lines.append("")

    lines.append('wait')
    lines.append('echo "All ColabFold jobs finished."')

    with open(path, "w") as f:
        f.write("\n".join(lines) + "\n")
    path.chmod(0o755)


def parse_args():
    parser = argparse.ArgumentParser(
        description="Generate ColabFold jobs (public API mode) for the gingipain "
                     "inhibitor pipeline, with an optional test-size limit."
    )
    parser.add_argument(
        "--limit", type=str, default=None,
        help="Number of candidates per track to include (e.g. '3' for a quick test). "
             "Pass 'all' to include every candidate in both tracks (only recommended "
             "once you've switched to local databases -- see module docstring). "
             "If omitted, uses CONFIG['candidates_per_track_limit'].",
    )
    return parser.parse_args(args=[])


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------
def main():
    args = parse_args()
    cfg = CONFIG

    if args.limit is not None:
        if args.limit.strip().lower() == "all":
            cfg["candidates_per_track_limit"] = None
        else:
            try:
                cfg["candidates_per_track_limit"] = int(args.limit)
            except ValueError:
                print(f"ERROR: --limit must be an integer or 'all', got '{args.limit}'")
                return

    limit = cfg["candidates_per_track_limit"]
    print(f"Mode: public API  |  candidates-per-track limit: "
          f"{'ALL (uncapped)' if limit is None else limit}")

    cfg["output_dir"].mkdir(parents=True, exist_ok=True)

    for key in ("variant_candidates_fasta", "database_candidates_fasta", "targets_fasta"):
        if not cfg[key].exists():
            print(f"ERROR: {cfg[key]} not found. Check the path in CONFIG.")
            return

    targets = read_fasta(cfg["targets_fasta"])
    print(f"Loaded {len(targets)} targets from {cfg['targets_fasta']}")
    if len(targets) != 3:
        print(f"WARNING: expected 3 targets (RgpA/RgpB/Kgp), found {len(targets)} -- "
              f"double check {cfg['targets_fasta']} before proceeding.")

    variant_candidates = read_fasta(cfg["variant_candidates_fasta"])
    database_candidates = read_fasta(cfg["database_candidates_fasta"])
    print(f"Loaded {len(variant_candidates)} variant-track candidates")
    print(f"Loaded {len(database_candidates)} database-track candidates")

    manifest_rows = []
    variant_jobs = generate_track_jobs(
        variant_candidates, targets, "variant", limit, manifest_rows
    )
    database_jobs = generate_track_jobs(
        database_candidates, targets, "database", limit, manifest_rows
    )

    variant_fasta_path = cfg["output_dir"] / "colabfold_input_variant.fasta"
    database_fasta_path = cfg["output_dir"] / "colabfold_input_database.fasta"
    write_jobs_fasta(variant_jobs, variant_fasta_path)
    write_jobs_fasta(database_jobs, database_fasta_path)

    # Manifest
    manifest_path = cfg["output_dir"] / "job_manifest.tsv"
    with open(manifest_path, "w") as f:
        f.write("job_id\tcandidate_accession\ttarget_id\tsource_track\t"
                "candidate_length\ttarget_length\n")
        for row in manifest_rows:
            f.write(f"{row['job_id']}\t{row['candidate_accession']}\t{row['target_id']}\t"
                    f"{row['source_track']}\t{row['candidate_length']}\t{row['target_length']}\n")

    # Launch script
    launch_path = cfg["output_dir"] / "run_colabfold.sh"
    write_launch_script(
        cfg, launch_path,
        [("variant", variant_fasta_path), ("database", database_fasta_path)],
    )

    total_jobs = len(variant_jobs) + len(database_jobs)
    print(f"\nDone.")
    print(f"  Variant track jobs:  {len(variant_jobs)}  -> {variant_fasta_path}")
    print(f"  Database track jobs: {len(database_jobs)} -> {database_fasta_path}")
    print(f"  Total jobs: {total_jobs}")
    print(f"  Manifest: {manifest_path}")
    print(f"  Launch script: {launch_path}")
    print(f"\nNEXT STEPS:")
    if limit is not None:
        print(f"  1. This is a TEST slice ({limit} candidates/track, {total_jobs} jobs total).")
        print(f"     Confirm colabfold_batch is installed and results look right, then")
        print(f"     scale up gradually: --limit 50, --limit 200, etc.")
        print(f"     Do NOT jump straight to --limit all against the public API --")
        print(f"     switch to local databases (setup_databases.sh + colabfold_search)")
        print(f"     before running the full ~9,810-job set.")
    else:
        print(f"  1. Running UNCAPPED ({total_jobs} jobs) -- if this is still hitting the")
        print(f"     public API rather than local databases, this is likely too large")
        print(f"     for the shared server. Confirm you've set up local databases first.")
    print(f"  2. Confirm colabfold_batch is installed and on PATH: colabfold_batch --help")
    print(f"  3. Run: bash {launch_path}")
    print(f"  4. Once results land in output/colabfold_jobs/results_*/, the next")
    print(f"     script (parse_colabfold_results.py, not yet written) will collect")
    print(f"     iptm/plddt scores per job and join them back to job_manifest.tsv.")


if __name__ == "__main__":
    main()

Mode: public API  |  candidates-per-track limit: 3
Loaded 3 targets from output/targets/catalytic_domains.fasta
Loaded 1158 variant-track candidates
Loaded 2112 database-track candidates

Done.
  Variant track jobs:  9  -> output/colabfold_jobs/colabfold_input_variant.fasta
  Database track jobs: 9 -> output/colabfold_jobs/colabfold_input_database.fasta
  Total jobs: 18
  Manifest: output/colabfold_jobs/job_manifest.tsv
  Launch script: output/colabfold_jobs/run_colabfold.sh

NEXT STEPS:
  1. This is a TEST slice (3 candidates/track, 18 jobs total).
     Confirm colabfold_batch is installed and results look right, then
     scale up gradually: --limit 50, --limit 200, etc.
     Do NOT jump straight to --limit all against the public API --
     switch to local databases (setup_databases.sh + colabfold_search)
     before running the full ~9,810-job set.
  2. Confirm colabfold_batch is installed and on PATH: colabfold_batch --help
  3. Run: bash output/colabfold_jobs/run_colabfold.sh
  4

In [3]:
!bash output/colabfold_jobs/run_colabfold.sh

=== variant track ===
=== database track ===
All ColabFold jobs finished.
